# Algorithms and Processes of a No-Limit Texas Hold'em Engine

### Annotated Python notebook: theory, mathematics, algorithms, and verification

In this notebook, we turn our `no_limit_holdem_engine_algorithms.py` reference implementation into a guided, executable study of a complete poker engine. We follow the structure of our companion report, *Algorithms and Processes of a No-Limit Hold'em Engine*, and build every component from the ground up.

Our goal is to explain not only **what** each function computes, but **why** the algorithm is designed the way it is, what mathematics it relies on, how edge cases are handled in passive and active stages of a hand, and which checks verify its correctness.

> **How to use this notebook:** Run cells from top to bottom. Only the Python standard library is required. Cells that use our seeded random generator (`random.Random(42)`) give reproducible Monte Carlo results, while exact enumeration cells are deterministic with zero sampling error ($\mathrm{SE} = 0$).

**Source note.** Every code cell in this notebook is synchronized with our $4$ to $8$ player engine implementation. Where a heuristic or simplified solver demonstration is used alongside an exact lookup table, we state its scope plainly in standard language.

## Learning map

| Stage | What we cover | Main mathematics and algorithms |
|---|---|---|
| 1. Foundations | Rules, combinations, expected value, pot odds, MDF, and bluff fractions | $\binom{n}{k}$, $\mathbb{E}[X]$, break-even inequalities |
| 2. Cards and randomness | Compact card IDs, fair shuffling, and our sealed dealer vault | Bitwise shifts, rejection sampling, Fisher-Yates, SHA-256 seal |
| 3. Hand evaluation | How five-card and seven-card hands are ranked | Histograms, base-$13$ scores, $13$-bit masks, lookup tables |
| 4. Equity | How likely a hand is to win a share of the pot | Conditional probability, Bessel ($N - 1$) Monte Carlo $\mathrm{SE}$, multi-way bunching, exact enumeration |
| 5. Betting and pots | How $4$ to $8$ player turns, raises, all-ins, and chips are managed | State transitions, TDA Rule 19 blinds, chip conservation, horizontal contribution bands |
| 6. AI and adaptation | How actions are chosen, short stacks are solved, and opponents are modeled | Chen valuation, $169 \times 19$ Nash push/fold table, CFR, Malmuth-Harville ICM, Beta-Binomial updating |
| 7. Verification | How all seven formal invariants are checked on a complete hand | Invariants $I_1$ through $I_7$, differential tests, worked $4$-player hand |

### A useful distinction

In our architecture, four layers are kept separate. A **rules engine** decides which actions are legal and where chips move. A **hand evaluator** compares five-card and seven-card hands. An **equity calculator** estimates expected showdown pot share. A **strategy policy** chooses actions under imperfect information. Keeping these layers distinct lets us test each one independently.

## 1. Foundations: poker, probability, and decisions

### 1.1 The hand as a sequence of public and private information

Texas Hold'em is played with a $52$-card deck. Each player is dealt two private *hole cards*. Up to five community cards are revealed across the flop (three cards), the turn (one card), and the river (one card). At showdown, each remaining player forms the best five-card poker hand available from their two hole cards and the five-card board.

A player may use both hole cards, one hole card, or neither. When the five-card board itself forms the best possible hand for every remaining player, the board plays and the pot is split.

### 1.2 Counting outcomes with combinations

The number of ways to select $k$ distinct cards from $n$ when order does not matter is given by the binomial coefficient:

$$
\binom{n}{k} = \frac{n!}{k!\,(n - k)!}.
$$

For example, there are

$$
\binom{52}{2} = \frac{52 \cdot 51}{2} = 1{,}326
$$

two-card starting combinations, and

$$
\binom{52}{5} = \frac{52 \cdot 51 \cdot 50 \cdot 49 \cdot 48}{5 \cdot 4 \cdot 3 \cdot 2 \cdot 1} = 2{,}598{,}960
$$

distinct five-card hands. Seven available cards contain $\binom{7}{5} = 21$ different five-card subsets, which is why our reference evaluator can check all $21$ subsets and keep the highest score.

**Hand-category convention:** our combinatorics table lists Royal Flush separately from other Straight Flushes, giving ten display categories ($\kappa \in \{0, \ldots, 9\}$). Inside our packed evaluator, nine numeric categories ($\kappa \in \{0, \ldots, 8\}$, High Card through Straight Flush) are used, and a Royal Flush is naturally represented as an Ace-high Straight Flush ($\kappa = 8$, top rank $12$). Both conventions produce the exact same strict hand ordering.

### 1.3 Expected value and pot odds

Expected value ($\mathrm{EV}$) is the probability-weighted average of all possible payoffs. We use it to compare decisions over many repetitions rather than to judge a single outcome in hindsight.

Let $P$ be the pot **after the opponent's bet but before our call**, let $C$ be the additional call amount required from us, and let $q \in [0, 1]$ be our expected share of the final pot. When no further betting occurs after the call, our expected value is:

$$
\mathrm{EV}(\text{call}) = q(P + C) - C.
$$

Setting $\mathrm{EV}(\text{call}) = 0$ gives the break-even equity threshold:

$$
q^* = \frac{C}{P + C}.
$$

**Example.** Suppose the pot is $P = 150$ after an opponent bets, and calling costs $C = 50$. The final pot after we call is $P + C = 200$, so $q^* = 50 / 200 = 25\%$. Any call with equity $q > 25\%$ has positive chip $\mathrm{EV}$ under this single-street model.

### 1.4 MDF, bluff fractions, and SPR

When a bettor wagers $B$ into a pot $P_0$, a pure bluff risks $B$ to win $P_0$. The break-even fold frequency for that bluff is $B / (P_0 + B)$. Its complement gives the *minimum defense frequency* ($\mathrm{MDF}$):

$$
\mathrm{MDF} = \frac{P_0}{P_0 + B}.
$$

For a polarized river betting range composed of value hands and bluffs, the balanced bluff fraction that makes a bluff-catcher indifferent between calling and folding is:

$$
\alpha^*_{\text{bluff}} = \frac{B}{P_0 + 2B}.
$$

The **stack-to-pot ratio** ($\mathrm{SPR}$) is the effective remaining stack $S_{\mathrm{eff}}$ divided by the flop pot $P_{\mathrm{flop}}$:

$$
\mathrm{SPR} = \frac{S_{\mathrm{eff}}}{P_{\mathrm{flop}}}.
$$

A low $\mathrm{SPR}$ ($\mathrm{SPR} \le 3$) means a single pot-sized bet and call commits the remaining stack.

### Code: foundational equations and helper functions

In the cell below, we translate these foundational equations into small helper functions. All pot and bet arguments use the same chip units.

In [1]:
import hashlib
import json as _json
import math
import os
import random
import secrets
import time
from dataclasses import dataclass, field
from itertools import combinations
from typing import Callable, Dict, List, Optional, Set, Tuple

def expected_value_call(equity, pot_before_call, call_cost):
    "Compute EV(Call) = q * (P + C) - C."
    return equity * (pot_before_call + call_cost) - call_cost

def pot_odds_threshold(pot_before_call, call_cost):
    "Compute the break-even equity threshold q* = C / (P + C)."
    if call_cost <= 0:
        return 0.0
    return call_cost / (pot_before_call + call_cost)

def minimum_defense_frequency(pot_before_bet, bet_size):
    "Compute MDF = P_0 / (P_0 + B) so an opponent's pure bluff has non-positive EV."
    return pot_before_bet / (pot_before_bet + bet_size)

def optimal_bluff_fraction(pot_before_bet, bet_size):
    "Compute the balanced bluff share B / (P_0 + 2B) inside a polarized river bet."
    return bet_size / (pot_before_bet + 2.0 * bet_size)

def stack_to_pot_ratio(effective_stack, flop_pot):
    "Compute SPR = S_eff / P_flop."
    return effective_stack / max(1.0, flop_pot)

### Code: exact five-card combinatorics and a bet-size reference table

Each category count below is derived from elementary combinations of ranks and suits. Our assertion verifies that the ten display categories partition all $\binom{52}{5} = 2{,}598{,}960$ five-card hands without overlap or omission.

In [2]:
# Here we verify the exact combinatorial counts for all 10 five-card hand categories.
comb = math.comb
CATEGORY_COMBINATORICS = [
    (9, "Royal Flush",     comb(4, 1)),
    (8, "Straight Flush",  9 * comb(4, 1)),
    (7, "Four of a Kind",  comb(13, 1) * comb(4, 4) * comb(12, 1) * comb(4, 1)),
    (6, "Full House",      comb(13, 1) * comb(4, 3) * comb(12, 1) * comb(4, 2)),
    (5, "Flush",           comb(4, 1) * comb(13, 5) - 40),
    (4, "Straight",        10 * (4**5 - 4)),
    (3, "Three of a Kind", comb(13, 1) * comb(4, 3) * comb(12, 2) * (comb(4, 1)**2)),
    (2, "Two Pair",        comb(13, 2) * (comb(4, 2)**2) * comb(11, 1) * comb(4, 1)),
    (1, "One Pair",        comb(13, 1) * comb(4, 2) * comb(12, 3) * (comb(4, 1)**3)),
    (0, "High Card",       (comb(13, 5) - 10) * (4**5 - 4)),
]

total_5card = comb(52, 5)
assert sum(cnt for _, _, cnt in CATEGORY_COMBINATORICS) == total_5card

print(f"{'Cat':>3} | {'Hand Category':<16} | {'Exact Count':>11} | {'Probability':>11}")
for cat_id, name, count in CATEGORY_COMBINATORICS:
    print(f"{cat_id:>3} | {name:<16} | {count:>11,d} | {100.0 * count / total_5card:>10.6f}%")
print(f"{'All':>3} | {'Total C(52,5)':<16} | {total_5card:>11,d} | {100.0:>10.6f}%\n")

print("Game-Theoretic Bet-Sizing Reference Table (Pot P_0 = 100):")
print(f"{'Bet Size':<14} | {'Pot Odds q*':>12} | {'MDF':>10} | {'Optimal Bluff %':>15}")
for label, b in [("33% Pot (33)", 33.0), ("50% Pot (50)", 50.0), ("75% Pot (75)", 75.0), ("100% Pot (100)", 100.0), ("150% Overbet", 150.0)]:
    q_star = pot_odds_threshold(100.0 + b, b)
    mdf = minimum_defense_frequency(100.0, b)
    bluff_pct = optimal_bluff_fraction(100.0, b)
    print(f"{label:<14} | {100*q_star:>11.2f}% | {100*mdf:>9.2f}% | {100*bluff_pct:>14.2f}%")

Cat | Hand Category    | Exact Count | Probability
  9 | Royal Flush      |           4 |   0.000154%
  8 | Straight Flush   |          36 |   0.001385%
  7 | Four of a Kind   |         624 |   0.024010%
  6 | Full House       |       3,744 |   0.144058%
  5 | Flush            |       5,108 |   0.196540%
  4 | Straight         |      10,200 |   0.392465%
  3 | Three of a Kind  |      54,912 |   2.112845%
  2 | Two Pair         |     123,552 |   4.753902%
  1 | One Pair         |   1,098,240 |  42.256903%
  0 | High Card        |   1,302,540 |  50.117739%
All | Total C(52,5)    |   2,598,960 | 100.000000%

Game-Theoretic Bet-Sizing Reference Table (Pot P_0 = 100):
Bet Size       |  Pot Odds q* |        MDF | Optimal Bluff %
33% Pot (33)   |       19.88% |     75.19% |          19.88%
50% Pot (50)   |       25.00% |     66.67% |          25.00%
75% Pot (75)   |       30.00% |     57.14% |          30.00%
100% Pot (100) |       33.33% |     50.00% |          33.33%
150% Overbet   |       

## 2. Process I: compact card representation, fair shuffling, and dealer enclave sealing

### 2.1 Turning a card into an integer

Rather than store each card as a dictionary or object, we encode every card as a single integer $c \in \{0, \ldots, 51\}$:

$$
c = 4\rho + \sigma, \qquad \rho(c) = \lfloor c / 4 \rfloor = c \gg 2, \qquad \sigma(c) = c \bmod 4 = c \mathbin{\&} 3,
$$

where $\rho \in \{0, \ldots, 12\}$ is the rank index ($2$ through Ace) and $\sigma \in \{0, 1, 2, 3\}$ is the suit index ($\spadesuit, \heartsuit, \diamondsuit, \clubsuit$).

Because $4 = 2^2$, the rank is recovered with `c >> 2` (which drops the two suit bits), and the suit is recovered with `c & 3` (which keeps only the two lowest bits). For example, the Ace of Hearts has rank index $\rho = 12$ and suit index $\sigma = 1$, so its integer ID is $4(12) + 1 = 49$.

### Code: card IDs, parsing, and display

The helper functions below convert between human-readable card strings like `As` or `A♠` and our integer encoding $c \in \{0, \ldots, 51\}$.

In [3]:
RANK_CHARS = "23456789TJQKA"
RANK_NAMES = [
    "Two", "Three", "Four", "Five", "Six", "Seven", "Eight",
    "Nine", "Ten", "Jack", "Queen", "King", "Ace"
]
RANK_PLURAL = [
    "Twos", "Threes", "Fours", "Fives", "Sixes", "Sevens", "Eights",
    "Nines", "Tens", "Jacks", "Queens", "Kings", "Aces"
]
SUIT_CHARS = "shdc"
SUIT_SYMBOLS = "♠♥♦♣"
FULL_DECK = tuple(range(52))

def encode_card(rank, suit):
    "Combine rank in 0..12 and suit in 0..3 into a single integer c = 4 * rank + suit."
    return (rank << 2) | suit

def card_rank(c):
    "Extract the rank index rho(c) = c >> 2 in 0..12."
    return c >> 2

def card_suit(c):
    "Extract the suit index sigma(c) = c & 3 in 0..3."
    return c & 3

def parse_card(token):
    "Parse shorthand like 'As', 'Kh', 'Td', or '5c' into an integer 0..51."
    token = token.strip()
    r = RANK_CHARS.index(token[0].upper())
    s_char = token[1].lower()
    s = SUIT_CHARS.index(s_char) if s_char in SUIT_CHARS else SUIT_SYMBOLS.index(token[1])
    return encode_card(r, s)

def parse_cards(tokens):
    "Parse space-separated card strings like 'As Kh Qd Jc Ts'."
    return [parse_card(t) for t in tokens.split() if t.strip()]

def card_str(c):
    "Format an integer card 0..51 as rank and suit symbol."
    return f"{RANK_CHARS[c >> 2]}{SUIT_SYMBOLS[c & 3]}"

def hand_str(cards):
    return " ".join(card_str(c) for c in cards)

### 2.2 Why we shuffle with rejection-sampled Fisher-Yates and seal the deck in `DealerVault`

A fair shuffle must give every one of the $52!$ deck permutations the exact same probability $1 / 52!$. In the Fisher-Yates shuffle, we walk backward from index $i = 51$ down to $1$: at index $i$, we pick one of the $i + 1$ remaining positions uniformly and swap it into place.

When we draw a $32$-bit random word $x \in [0, 2^{32})$ and reduce it modulo $n$, naive `x % n` introduces modulo bias whenever $n$ does not divide $2^{32}$. We remove that bias via rejection sampling:

$$
L = \left\lfloor \frac{2^{32}}{n} \right\rfloor n.
$$

Any sample $x \ge L$ is rejected and redrawn. Within the accepted range $[0, L)$, every remainder modulo $n$ appears the exact same number of times $\lfloor 2^{32} / n \rfloor$.

In addition, to prevent undealt stub cards or hidden opponent hole cards from sitting exposed on a global state object, we encapsulate the live deck and private hole cards inside a closure-isolated enclave (`create_dealer_vault`, Algorithm 21). Before the first card is dealt, a $128$-bit cryptographic salt is generated and a SHA-256 **Provably Fair commitment seal** is computed over the shuffled $52$-card order and salt, which can be verified after showdown.

### Algorithm 1 and Algorithm 21: unbiased shuffling and our closure-sealed dealer vault

The cell below implements `secure_random_int`, `shuffle_deck_in_place`, and `create_dealer_vault`, then verifies the $52$-card bijection and SHA-256 commitment seal.

In [4]:
# We use rejection sampling over 32-bit OS entropy so modulo reduction never biases the shuffle.
def secure_random_int(n):
    "Return an unbiased random integer in 0..n-1 using rejection sampling."
    if n <= 1:
        return 0
    two32 = 1 << 32
    limit = (two32 // n) * n
    while True:
        x = int.from_bytes(os.urandom(4), "little")
        if x < limit:
            return x % n

def shuffle_deck_in_place(deck):
    "Permute the deck in place using the Fisher-Yates shuffle."
    for i in range(len(deck) - 1, 0, -1):
        j = secure_random_int(i + 1)
        deck[i], deck[j] = deck[j], deck[i]
    return deck

# In our online-hardened design, undealt cards and private hole cards live inside a closure
# rather than on a public state dictionary, and a salted SHA-256 seal commits to the shuffle.
def create_dealer_vault():
    "Create a closure-isolated dealer enclave with a salted SHA-256 Provably Fair commitment seal."
    _deck = []
    _hole_store = {}
    _burned = []
    _salt = ""
    _seal = ""
    _initial_order = []

    def init_hand(dealt_seat_ids, sb_seat, n_seats, custom_deck=None):
        nonlocal _deck, _hole_store, _burned, _salt, _seal, _initial_order
        _deck = list(custom_deck) if custom_deck is not None else shuffle_deck_in_place(list(FULL_DECK))
        _initial_order = list(_deck)
        _hole_store = {}
        _burned = []
        _salt = secrets.token_hex(16)
        payload = ":".join(card_str(c) for c in _initial_order) + "|" + _salt
        _seal = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:32]

        dealt_set = set(dealt_seat_ids)
        temp_holes = {sid: [] for sid in dealt_seat_ids}
        for _ in range(2):
            for i in range(n_seats):
                seat_idx = (sb_seat + i) % n_seats
                if seat_idx in dealt_set:
                    temp_holes[seat_idx].append(_deck.pop())
        for sid, cards in temp_holes.items():
            _hole_store[sid] = (cards[0], cards[1])
        return _seal

    def get_hole(seat_id):
        pair = _hole_store.get(seat_id)
        return [pair[0], pair[1]] if pair is not None else []

    def deal_street(count):
        if _deck:
            _burned.append(_deck.pop())
        return [_deck.pop() for _ in range(count)]

    def verify_seal():
        payload = ":".join(card_str(c) for c in _initial_order) + "|" + _salt
        recomputed = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:32]
        assert recomputed == _seal, "Provably fair seal mismatch!"
        return {"seal": _seal, "salt": _salt, "verified": "PASS"}

    def audit_bijection(board):
        all_cards = list(_deck) + list(_burned) + list(board)
        for pair in _hole_store.values():
            all_cards.extend(pair)
        return len(all_cards) == 52 and set(all_cards) == set(FULL_DECK)

    return {
        "init_hand": init_hand,
        "get_hole": get_hole,
        "deal_street": deal_street,
        "verify_seal": verify_seal,
        "audit_bijection": audit_bijection,
    }

assert all(encode_card(card_rank(c), card_suit(c)) == c for c in range(52))
sample_deck = list(FULL_DECK)
shuffle_deck_in_place(sample_deck)
assert len(sample_deck) == 52 and set(sample_deck) == set(FULL_DECK)

vault = create_dealer_vault()
demo_seal = vault["init_hand"]([0, 1, 2, 3], sb_seat=1, n_seats=4)
flop_demo = vault["deal_street"](3)
assert vault["audit_bijection"](flop_demo) is True
assert vault["verify_seal"]()["verified"] == "PASS"

print("Bijective Card Encoding Verification: PASS (all 52 cards map 1 to 1)")
print("Sample Cryptographically Shuffled Deck (first 13 cards):", hand_str(sample_deck[:13]))
print(f"DealerVault Provably Fair Commitment Seal: #{demo_seal} (52-card bijection verified)")

Bijective Card Encoding Verification: PASS (all 52 cards map 1 to 1)
Sample Cryptographically Shuffled Deck (first 13 cards): K♦ Q♣ 5♠ K♠ J♦ 2♥ 2♦ 3♣ 8♣ 6♦ 7♣ J♣ K♣
DealerVault Provably Fair Commitment Seal: #138bc99e8fc02d4932cc97d532a48fe1 (52-card bijection verified)


## 3. Process II: hand evaluation

### 3.1 Encoding a hand so integer comparison gives exact poker order

When two hands are compared, the hand category is checked first; only if both hands belong to the same category do we compare their tie-breaker ranks (and kickers) from most significant to least significant.

To avoid multi-branch comparisons during simulation, we pack the category $\kappa \in \{0, \ldots, 8\}$ and up to five tie-break digits $k_i \in \{0, \ldots, 12\}$ into a single base-$13$ integer:

$$
V(\kappa, k) = \kappa \cdot 13^5 + \sum_{i=0}^{4} k_i \cdot 13^{4 - i}, \qquad 13^5 = 371{,}293.
$$

Because every rank digit satisfies $0 \le k_i \le 12$, the maximum possible sum of all lower-significance positions satisfies

$$
\sum_{i=j+1}^{4} 12 \cdot 13^{4 - i} = 13^{4 - j} - 1 < 13^{4 - j},
$$

so no lower-significance kicker can ever spill over into a higher-significance position. Unpacking reverses the operation by taking remainders and integer division by $13$.

### Algorithms 2 and 6: pack and unpack a base-$13$ score

In [5]:
# We pack the category and up to five tie-breaker ranks into one base-13 integer
# so any two hands can be compared with a single integer comparison.
def encode_score(category, tie_breakers):
    "Pack category in 0..8 and up to 5 kickers into a base-13 integer."
    v = category
    for i in range(5):
        v = v * 13 + (tie_breakers[i] if i < len(tie_breakers) else 0)
    return v

def decode_score(v):
    "Unpack a base-13 integer score into (category, [k0, k1, k2, k3, k4])."
    ks = [0] * 5
    x = v
    for i in range(4, -1, -1):
        ks[i] = x % 13
        x //= 13
    return x, ks

def format_hand_name(v):
    "Convert a packed base-13 score into a readable English description."
    cat, k = decode_score(v)
    if cat == 8:
        return "Royal Flush" if k[0] == 12 else f"Straight Flush, {RANK_NAMES[k[0]]} high"
    if cat == 7:
        return f"Four of a Kind, {RANK_PLURAL[k[0]]}"
    if cat == 6:
        return f"Full House, {RANK_PLURAL[k[0]]} full of {RANK_PLURAL[k[1]]}"
    if cat == 5:
        return f"Flush, {RANK_NAMES[k[0]]} high"
    if cat == 4:
        return f"Straight, {RANK_NAMES[k[0]]} high"
    if cat == 3:
        return f"Three of a Kind, {RANK_PLURAL[k[0]]}"
    if cat == 2:
        return f"Two Pair, {RANK_PLURAL[k[0]]} and {RANK_PLURAL[k[1]]}"
    if cat == 1:
        return f"Pair of {RANK_PLURAL[k[0]]}"
    return f"High Card, {RANK_NAMES[k[0]]}"

### 3.2 Classifying exactly five cards (`eval5`)

Our reference five-card evaluator counts how often each rank appears in a $13$-element histogram, checks whether all five suits match, and groups present ranks by multiplicity ($4$, $3$, $2$, or $1$). It then tests categories from strongest to weakest:

1. Straight Flush ($\kappa = 8$)
2. Four of a Kind ($\kappa = 7$)
3. Full House ($\kappa = 6$)
4. Flush ($\kappa = 5$)
5. Straight ($\kappa = 4$)
6. Three of a Kind ($\kappa = 3$)
7. Two Pair ($\kappa = 2$)
8. One Pair ($\kappa = 1$)
9. High Card ($\kappa = 0$)

When all five ranks are distinct (`len(uniq) == 5`), five consecutive ranks satisfy $U_0 - U_4 = 4$. The Ace-low wheel ($\text{A}, 2, 3, 4, 5$) has rank indices $(12, 3, 2, 1, 0)$ and is assigned top rank $s_{\mathrm{top}} = 3$ (Five high) so it ranks strictly below a Six-high straight ($s_{\mathrm{top}} = 4$).

### Algorithm 3: five-card classifier (`eval5`)

In [6]:
# This reference classifier scores any 5-card hand directly from its rank histogram and suit parity.
def eval5(cards5):
    "Classify and score an exact 5-card hand."
    s0 = cards5[0] & 3
    flush = all((c & 3) == s0 for c in cards5)

    cnt = [0] * 13
    for c in cards5:
        cnt[c >> 2] += 1

    # Collect distinct present ranks in descending order.
    uniq = [r for r in range(12, -1, -1) if cnt[r] > 0]
    groups = {1: [], 2: [], 3: [], 4: []}
    for r in uniq:
        groups[cnt[r]].append(r)

    straight = False
    s_top = -1
    if len(uniq) == 5:
        if uniq[0] - uniq[4] == 4:
            straight = True
            s_top = uniq[0]
        elif uniq[0] == 12 and uniq[1] == 3 and uniq[4] == 0:
            # Ace-low wheel straight (A, 2, 3, 4, 5) is ranked as 5-high (rank index 3).
            straight = True
            s_top = 3

    if straight and flush:
        return encode_score(8, [s_top])
    if groups[4]:
        return encode_score(7, [groups[4][0], groups[1][0]])
    if groups[3] and groups[2]:
        return encode_score(6, [groups[3][0], groups[2][0]])
    if flush:
        return encode_score(5, uniq)
    if straight:
        return encode_score(4, [s_top])
    if groups[3]:
        return encode_score(3, [groups[3][0], groups[1][0], groups[1][1]])
    if len(groups[2]) == 2:
        return encode_score(2, [groups[2][0], groups[2][1], groups[1][0]])
    if len(groups[2]) == 1:
        return encode_score(1, [groups[2][0], groups[1][0], groups[1][1], groups[1][2]])
    return encode_score(0, uniq)

### 3.3 Best five cards from seven (`best7`)

At showdown, up to seven cards are available to each player: two hole cards plus five board cards. Because any five of those seven cards may form the hand, there are

$$
\binom{7}{5} = \frac{7!}{5!\,2!} = 21
$$

subsets to check. `best7` evaluates every five-card subset and returns both the highest score $V^*$ and the five-card subset $C^*$ that achieved it. We use `best7` at showdown to highlight the winning five cards and as our ground-truth oracle when testing faster evaluators.

In [7]:
# At showdown we want both the best score and the exact 5 cards that produced it.
def best7(cards):
    "Evaluate all C(n, 5) subsets and return (best_score, best_5_cards)."
    best_score = -1
    best_subset = ()
    for sub in combinations(cards, 5):
        sc = eval5(sub)
        if sc > best_score:
            best_score = sc
            best_subset = sub
    return best_score, best_subset

### 3.4 Fast seven-card evaluator (`rank7fast`)

Inside a Monte Carlo equity rollout, calling `eval5` $21$ times per seven-card hand creates unnecessary overhead. Our fast evaluator (`rank7fast`) computes the exact same base-$13$ score in a single pass over the seven cards by accumulating:

* a $13$-bit mask of all ranks present (`all_mask`);
* one $13$-bit rank mask per suit ($s_0$, $s_1$, $s_2$, $s_3$);
* a $13$-element rank count array (`cnt`).

Three small $8{,}192$-entry lookup tables indexed by a $13$-bit rank mask $m \in \{0, \ldots, 2^{13} - 1\}$ provide the number of set bits ($\mathrm{POPC}[m]$), the top rank of any straight ($\mathrm{STR8}[m]$), and the packed base-$13$ value of the top five ranks ($\mathrm{TOP5T}[m]$). By the Pigeonhole Principle, at most one suit can have five or more cards in a seven-card hand, and when a five-card flush exists, neither Four of a Kind nor a Full House is possible.

### Algorithm 5: lookup tables and `rank7fast`

In [8]:
# We precompute three 8,192-entry lookup tables indexed by any 13-bit rank mask.
POPC = [0] * 8192
STR8 = [-1] * 8192
TOP5T = [0] * 8192

for m in range(8192):
    POPC[m] = m.bit_count()
    for r in range(8, -1, -1):
        if (m & (31 << r)) == (31 << r):
            STR8[m] = r + 4
            break
    if STR8[m] < 0 and (m & 0x100F) == 0x100F:
        STR8[m] = 3
    top5 = []
    for r in range(12, -1, -1):
        if (m >> r) & 1:
            top5.append(r)
            if len(top5) == 5:
                break
    val = 0
    for i in range(5):
        val = val * 13 + (top5[i] if i < len(top5) else 0)
    TOP5T[m] = val

P13_5 = 371293
P13_4 = 28561
P13_3 = 2197
P13_2 = 169

def rank7fast(cards):
    "Compute the exact base-13 score of a 5, 6, or 7 card hand in a single pass."
    s0 = s1 = s2 = s3 = 0
    all_mask = 0
    cnt = [0] * 13
    for c in cards:
        r = c >> 2
        u = c & 3
        b = 1 << r
        all_mask |= b
        cnt[r] += 1
        if u == 0:
            s0 |= b
        elif u == 1:
            s1 |= b
        elif u == 2:
            s2 |= b
        else:
            s3 |= b

    # At most one suit can have 5 or more cards in a 7-card hand.
    fm = s0 if POPC[s0] >= 5 else (s1 if POPC[s1] >= 5 else (s2 if POPC[s2] >= 5 else (s3 if POPC[s3] >= 5 else 0)))
    if fm:
        sf = STR8[fm]
        if sf >= 0:
            return 8 * P13_5 + sf * P13_4
        return 5 * P13_5 + TOP5T[fm]

    q0 = t0 = t1 = p0 = p1 = p2 = -1
    for r in range(12, -1, -1):
        k = cnt[r]
        if k == 4:
            q0 = r
        elif k == 3:
            if t0 < 0:
                t0 = r
            else:
                t1 = r
        elif k == 2:
            if p0 < 0:
                p0 = r
            elif p1 < 0:
                p1 = r
            elif p2 < 0:
                p2 = r

    if q0 >= 0:
        kicker = next(r for r in range(12, -1, -1) if r != q0 and cnt[r] > 0)
        return 7 * P13_5 + q0 * P13_4 + kicker * P13_3

    if t0 >= 0 and (t1 >= 0 or p0 >= 0):
        pair_rank = max(t1, p0)
        return 6 * P13_5 + t0 * P13_4 + pair_rank * P13_3

    st = STR8[all_mask]
    if st >= 0:
        return 4 * P13_5 + st * P13_4

    if t0 >= 0:
        k0 = k1 = -1
        for r in range(12, -1, -1):
            if cnt[r] == 1:
                if k0 < 0:
                    k0 = r
                else:
                    k1 = r
                    break
        return 3 * P13_5 + t0 * P13_4 + k0 * P13_3 + k1 * P13_2

    if p1 >= 0:
        k0 = next(r for r in range(12, -1, -1) if r != p0 and r != p1 and cnt[r] > 0)
        return 2 * P13_5 + p0 * P13_4 + p1 * P13_3 + k0 * P13_2

    if p0 >= 0:
        k0 = k1 = k2 = -1
        for r in range(12, -1, -1):
            if cnt[r] == 1:
                if k0 < 0:
                    k0 = r
                elif k1 < 0:
                    k1 = r
                else:
                    k2 = r
                    break
        return P13_5 + p0 * P13_4 + k0 * P13_3 + k1 * P13_2 + k2 * 13

    return TOP5T[all_mask]

### 3.5 Related evaluator architectures

In our report, we also analyze two classic lookup-table evaluator mechanisms from the literature:

* **Prime-product rank multiset identity (Cactus Kev):** each of the $13$ ranks is assigned one of the first $13$ prime numbers $p_r \in \{2, 3, 5, \ldots, 41\}$. By the Fundamental Theorem of Arithmetic, the product $\prod_{i=1}^{5} p_{r_i}$ uniquely identifies any five-card rank multiset regardless of card order.
* **Quinary combinadic dynamic-programming hash (`PHEvaluator`):** a seven-card hand's rank histogram is a $13$-element vector $q \in \{0, 1, 2, 3, 4\}^{13}$ whose entries sum to $7$. The number of valid quinary vectors is

$$
N_{\mathrm{quinary}}(13, 7) = \binom{19}{7} - 13\binom{14}{2} = 50{,}388 - 1{,}183 = 49{,}205,
$$

and a precomputed dynamic-programming table maps all $49{,}205$ histograms bijectively into $\{0, \ldots, 49{,}204\}$ in $O(1)$ time.

### Associated evaluator structures and differential tests

In the next cell, we implement the prime-product helper and the quinary combinadic hash table, run a $5{,}000$-hand differential check verifying that `rank7fast` and `best7` return identical scores, and test canonical edge-case hands.

In [9]:
# First 13 prime numbers used by Cactus Kev's prime-product rank multiset encoding.
RANK_PRIMES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41)

def cactus_kev_prime_product(cards5):
    "Multiply the rank primes of 5 cards to uniquely identify their rank multiset."
    prod = 1
    for c in cards5:
        prod *= RANK_PRIMES[c >> 2]
    return prod

# Quinary combinadic dynamic programming table used by PHEvaluator to hash
# any 13-element rank histogram summing to 7 into 0..49204.
def build_quinary_dp_table():
    "Build the combinadic DP table mapping 7-card rank histograms into 0..49204."
    ways = [[0] * 8 for _ in range(14)]
    ways[0][0] = 1
    for r in range(1, 14):
        for k in range(8):
            ways[r][k] = sum(ways[r - 1][k - m] for m in range(min(4, k) + 1))

    dp = [[[0] * 8 for _ in range(13)] for _ in range(5)]
    for r in range(13):
        rem_ranks = 12 - r
        for l in range(5):
            for k in range(l, 8):
                dp[l][r][k] = sum(ways[rem_ranks][k - m] for m in range(l))
    return dp

QUINARY_DP = build_quinary_dp_table()

def hash_quinary_7(q):
    "Compute the bijective combinadic hash in 0..49204 for a 7-card rank histogram."
    h = 0
    rem = 7
    for r in range(13):
        qr = q[r]
        if qr > 0:
            h += QUINARY_DP[qr][r][rem]
            rem -= qr
    return h

# Differential test checking that rank7fast matches best7 on 5,000 random 7-card hands.
rng_deck = list(FULL_DECK)
mismatches = 0
for _ in range(5000):
    shuffle_deck_in_place(rng_deck)
    hand7 = rng_deck[:7]
    s_fast = rank7fast(hand7)
    s_best, best5_cards = best7(hand7)
    if s_fast != s_best:
        mismatches += 1
    cat, ks = decode_score(s_fast)
    assert encode_score(cat, ks) == s_fast

assert mismatches == 0
print(f"Differential Test (Rank7Fast vs Best7 over 5,000 random 7-card hands): {mismatches} mismatches!")
print(f"Quinary Combinadic DP Verification: N_quinary(13, 7) max hash + 1 = {hash_quinary_7([4, 3] + [0]*11) + 1:,d} (exact: 49,205)\n")

canonical_matchups = [
    ("Royal Flush vs King-High SF", "As Ks Qs Js Ts 2d 3c", "Ks Qs Js Ts 9s Ah Kd"),
    ("Six-High Straight vs Wheel",  "6s 5d 4h 3c 2s Ah Kd", "As 2d 3h 4c 5s Kh Qd"),
    ("Double-Trips Full House",     "Ks Kh Kd Qs Qh Qd 2c", "Ks Kh Kd Js Jh Jd Ac"),
]
for title, h1_str, h2_str in canonical_matchups:
    h1, h2 = parse_cards(h1_str), parse_cards(h2_str)
    v1, v2 = rank7fast(h1), rank7fast(h2)
    assert v1 > v2
    print(f"{title:<28}: {format_hand_name(v1)} ({v1:,d}) > {format_hand_name(v2)} ({v2:,d})")

Differential Test (Rank7Fast vs Best7 over 5,000 random 7-card hands): 0 mismatches!
Quinary Combinadic DP Verification: N_quinary(13, 7) max hash + 1 = 49,205 (exact: 49,205)

Royal Flush vs King-High SF : Royal Flush (3,313,076) > Straight Flush, King high (3,284,515)
Six-High Straight vs Wheel  : Straight, Six high (1,599,416) > Straight, Five high (1,570,855)
Double-Trips Full House     : Full House, Kings full of Queens (2,563,899) > Full House, Kings full of Jacks (2,561,702)


## 4. Process III: probability, equity, multi-way bunching, and runout calculations

### 4.1 Win probability versus pot-share equity

At showdown, a hand can win the entire pot, lose, or tie with one or more opponents and receive a fraction of the pot. Let $W \in [0, 1]$ denote the fraction of the pot won on a single simulated showdown ($1$ for an outright win, $0$ for a loss, $1/2$ for a two-way tie, $1/3$ for a three-way tie, and so on). Pot-share equity is the expected value:

$$
q = \mathbb{E}[W].
$$

Averaging fractional shares rather than binary wins ensures that split-pot hands are priced accurately when compared against pot odds.

### 4.2 Chen score: a fast starting-hand heuristic

Bill Chen's starting-hand formula converts any two hole cards into a single preflop strength number in $[-1, 20]$. High cards, pocket pairs, suitedness, and close connectivity are rewarded, while rank gaps are penalized. In our engine, we apply $\lceil s \rceil$ (`math.ceil`) at the end of the calculation (matching Chen's original rounding rule and our browser implementation) and precompute the $52 \times 52$ lookup matrix `CHEN_PAIR` so range filters can check a candidate hand in $O(1)$ time.

In [10]:
# Bill Chen's preflop hand valuation formula with ceiling rounding, plus our 52x52 lookup matrix.
def chen_score(c1, c2):
    "Compute Bill Chen's starting-hand strength score for two hole cards."
    r1, r2 = c1 >> 2, c2 >> 2
    s1, s2 = c1 & 3, c2 & 3
    hi, lo = max(r1, r2), min(r1, r2)
    base_map = {12: 10.0, 11: 8.0, 10: 7.0, 9: 6.0}
    score = base_map.get(hi, (hi + 2) / 2.0)

    if r1 == r2:
        return float(math.ceil(max(5.0, score * 2.0)))
    if s1 == s2:
        score += 2.0
    gap = hi - lo - 1
    if gap == 1:
        score -= 1.0
    elif gap == 2:
        score -= 2.0
    elif gap == 3:
        score -= 4.0
    elif gap >= 4:
        score -= 5.0
    if gap <= 1 and hi < 10:
        score += 1.0
    return float(math.ceil(score))

CHEN_PAIR = [[0.0] * 52 for _ in range(52)]
for a in range(52):
    for b in range(a + 1, 52):
        sc = chen_score(a, b)
        CHEN_PAIR[a][b] = sc
        CHEN_PAIR[b][a] = sc

### 4.3 Monte Carlo equity (`equity`) with Bessel-corrected standard error

When too many unseen cards remain for full enumeration, we estimate equity by simulating $N$ random deals from the unseen stub:

$$
\hat{q} = \frac{1}{N} \sum_{t=1}^{N} w_t.
$$

In each trial, only the first $k = b_{\mathrm{need}} + 2m$ positions of the stub array are swapped via partial Fisher-Yates, without resetting the stub array between trials. Because each trial outcome $w_t \in [0, 1]$ is a fractional pot share, we compute the unbiased **Bessel-corrected sample variance** (dividing by $N - 1$) and standard error:

$$
s^2 = \frac{1}{N - 1}\left(\sum_{t=1}^{N} w_t^2 - N\hat{q}^2\right), \qquad \mathrm{SE}(\hat{q}) = \sqrt{\frac{s^2}{N}}.
$$

In [11]:
# Monte Carlo equity rollout using in-place partial Fisher-Yates and Bessel's N-1 sample variance.
def equity(hole, board, num_opponents=1, iters=2000, rng=None):
    "Estimate pot-share equity, win/tie/loss probabilities, and unbiased standard error."
    if rng is None:
        rng = random
    dead = set(hole) | set(board)
    stub = [c for c in FULL_DECK if c not in dead]
    K = len(stub)
    b_need = 5 - len(board)
    k_draw = b_need + 2 * num_opponents

    sim_hero = [hole[0], hole[1]] + list(board) + [0] * b_need
    sim_opp = [0, 0] + list(board) + [0] * b_need
    b_len = len(board)

    w1 = 0.0
    w2 = 0.0
    wins = 0
    ties_cnt = 0
    losses = 0
    for _ in range(iters):
        for i in range(k_draw):
            j = i + int(rng.random() * (K - i))
            stub[i], stub[j] = stub[j], stub[i]

        for b in range(b_need):
            card_b = stub[b]
            sim_hero[2 + b_len + b] = card_b
            sim_opp[2 + b_len + b] = card_b

        v_hero = rank7fast(sim_hero)
        best_opp = -1
        ties = 0
        for p in range(num_opponents):
            sim_opp[0] = stub[b_need + 2 * p]
            sim_opp[1] = stub[b_need + 2 * p + 1]
            vp = rank7fast(sim_opp)
            if vp > best_opp:
                best_opp = vp
                ties = 1
            elif vp == best_opp:
                ties += 1

        if v_hero > best_opp:
            wt = 1.0
            wins += 1
        elif v_hero == best_opp:
            wt = 1.0 / (ties + 1)
            ties_cnt += 1
        else:
            wt = 0.0
            losses += 1
        w1 += wt
        w2 += wt * wt

    q_hat = w1 / iters
    var_s = max(0.0, (w2 - iters * q_hat * q_hat) / (iters - 1)) if iters > 1 else 0.0
    se = math.sqrt(var_s / iters)
    return {
        "equity": q_hat,
        "variance": var_s,
        "se": se,
        "win_prob": wins / iters,
        "tie_prob": ties_cnt / iters,
        "loss_prob": losses / iters,
    }

### 4.4 Range-conditioned equity (`equity_vs_range`), multi-way bunching (`equity_multiway_ranged`), and board-lock protection (`is_board_lock_nut_chop`)

Once an opponent voluntarily calls or raises preflop, they are much less likely to hold weak starting hands. We model this in three ways:

1. **Single-opponent range filtering (`equity_vs_range`):** candidate opponent hands with $\mathrm{Chen}(H_{\mathrm{opp}}) < c_{\mathrm{floor}}$ are rejected before drawing the remaining board cards.
2. **Multi-way range conditioning with preflop card-removal bunching (`equity_multiway_ranged`):** on $4$ to $8$ player tables, folded early-position hands are biased toward low cards ($\mathrm{Chen} \le 5$), enriching the remaining stub in high cards. We first draw bunched muck pairs for folded seats, then rejection-sample each active opponent against their own action-conditioned floor $c_{\mathrm{floor}, j}$.
3. **Board-lock split-pot guard (`is_board_lock_nut_chop`):** on a five-card river board, if the board itself is the unbeatable nuts ($\mathbb{P}(\text{loss}) = 0$), a player cannot lose the hand and should never fold a guaranteed split pot to an overbet.

In [12]:
# Single-opponent and multi-way range-conditioned Monte Carlo rollouts, plus board-lock detection.
def equity_vs_range(hole, board, chen_floor, iters=2000, rng=None):
    "Estimate conditional equity against a single opponent with Chen(H_opp) >= chen_floor."
    if rng is None:
        rng = random
    dead = set(hole) | set(board)
    stub = [c for c in FULL_DECK if c not in dead]
    K = len(stub)
    b_need = 5 - len(board)

    sim_hero = [hole[0], hole[1]] + list(board) + [0] * b_need
    sim_opp = [0, 0] + list(board) + [0] * b_need
    b_len = len(board)

    w1 = 0.0
    accepted = 0
    attempts = 0
    max_attempts = iters * 18

    while accepted < iters and attempts < max_attempts:
        attempts += 1
        j0 = int(rng.random() * K)
        stub[0], stub[j0] = stub[j0], stub[0]
        j1 = 1 + int(rng.random() * (K - 1))
        stub[1], stub[j1] = stub[j1], stub[1]

        if CHEN_PAIR[stub[0]][stub[1]] < chen_floor:
            continue

        for b in range(b_need):
            idx = 2 + b
            jb = idx + int(rng.random() * (K - idx))
            stub[idx], stub[jb] = stub[jb], stub[idx]
            sim_hero[2 + b_len + b] = stub[idx]
            sim_opp[2 + b_len + b] = stub[idx]

        sim_opp[0], sim_opp[1] = stub[0], stub[1]
        v_hero = rank7fast(sim_hero)
        v_opp = rank7fast(sim_opp)
        if v_hero > v_opp:
            w1 += 1.0
        elif v_hero == v_opp:
            w1 += 0.5
        accepted += 1

    return (w1 / accepted) if accepted >= 30 else None

def equity_multiway_ranged(hole, board, opp_floors, folded_count=0, iters=1200, rng=None):
    "Estimate multi-way range-conditioned equity with preflop card-removal bunching."
    if rng is None:
        rng = random
    n_opp = len(opp_floors)
    dead = set(hole) | set(board)
    stub = [c for c in FULL_DECK if c not in dead]
    L = len(stub)
    need = 5 - len(board)
    bunch_n = min(4, max(0, folded_count))
    if 2 * bunch_n + 2 * n_opp + need > L:
        return equity(hole, board, num_opponents=n_opp, iters=iters, rng=rng)

    tot = 0.0
    tot_sq = 0.0
    wins = 0
    ties_cnt = 0
    losses = 0
    done = 0
    guard = 0
    max_guard = iters * (60 if n_opp == 1 else 25)

    while done < iters and guard < max_guard:
        guard += 1
        ptr = 0
        # First we draw folded preflop hands biased toward low Chen scores (<= 5.0).
        for _ in range(bunch_n):
            for _ in range(3):
                j0 = ptr + int(rng.random() * (L - ptr))
                stub[ptr], stub[j0] = stub[j0], stub[ptr]
                j1 = ptr + 1 + int(rng.random() * (L - ptr - 1))
                stub[ptr + 1], stub[j1] = stub[j1], stub[ptr + 1]
                if CHEN_PAIR[stub[ptr]][stub[ptr + 1]] <= 5.0:
                    break
            ptr += 2

        # Next we sample each live opponent's hole cards against their range floor.
        all_ok = True
        for o in range(n_opp):
            fl = opp_floors[o]
            ok = False
            tries = 1 if n_opp == 1 else 16
            for _ in range(tries):
                j0 = ptr + int(rng.random() * (L - ptr))
                stub[ptr], stub[j0] = stub[j0], stub[ptr]
                j1 = ptr + 1 + int(rng.random() * (L - ptr - 1))
                stub[ptr + 1], stub[j1] = stub[j1], stub[ptr + 1]
                if CHEN_PAIR[stub[ptr]][stub[ptr + 1]] >= fl:
                    ok = True
                    break
            if not ok and n_opp == 1:
                all_ok = False
                break
            ptr += 2
        if not all_ok:
            continue

        runout_board = list(board)
        for b in range(need):
            idx = ptr + b
            jb = idx + int(rng.random() * (L - idx))
            stub[idx], stub[jb] = stub[jb], stub[idx]
            runout_board.append(stub[idx])

        mine = rank7fast(hole + runout_board)
        opp_start = 2 * bunch_n
        ties = 0
        lost = False
        for o in range(n_opp):
            r = rank7fast([stub[opp_start + 2 * o], stub[opp_start + 2 * o + 1]] + runout_board)
            if r > mine:
                lost = True
                break
            if r == mine:
                ties += 1
        if lost:
            losses += 1
            share = 0.0
        else:
            if ties > 0:
                ties_cnt += 1
            else:
                wins += 1
            share = 1.0 / (1 + ties)
        tot += share
        tot_sq += share * share
        done += 1

    mean = tot / max(1, done)
    var_s = max(0.0, (tot_sq - done * mean * mean) / max(1, done - 1)) if done > 1 else 0.0
    return {
        "equity": mean,
        "variance": var_s,
        "se": math.sqrt(var_s / max(1, done)),
        "win_prob": wins / max(1, done),
        "tie_prob": ties_cnt / max(1, done),
        "loss_prob": losses / max(1, done),
    }

def is_board_lock_nut_chop(hole, board5):
    "Return True when the 5-card river board itself is the unbeatable nuts."
    if len(board5) != 5 or len(hole) < 2:
        return False
    board_score = eval5(board5)
    if rank7fast(hole + board5) != board_score:
        return False
    dead = set(hole) | set(board5)
    stub = [c for c in FULL_DECK if c not in dead]
    for i in range(len(stub)):
        for j in range(i + 1, len(stub)):
            if rank7fast([stub[i], stub[j]] + board5) > board_score:
                return False
    return True

### 4.5 Exact river equity and the next-card scan

On a five-card river board, once our two hole cards are known, $45$ unseen cards remain. A single opponent can hold any unordered pair from those $45$ cards:

$$
\binom{45}{2} = \frac{45 \cdot 44}{2} = 990.
$$

Enumerating all $990$ pairs gives exact river equity with zero sampling error ($\mathrm{SE} = 0$). On a heads-up turn ($4$ board cards), $46$ candidate river cards remain, and each river leaves $990$ opponent holdings:

$$
46 \times \binom{45}{2} = 45{,}540 \text{ showdowns}.
$$

Our `next_card_scan` routine enumerates all $45{,}540$ deals on a heads-up turn ($\mathrm{SE} = 0$) and uses a dynamic $2\sigma$ threshold

$$
\tau_c = \max\!\left(0.03,\, 2\sqrt{\mathrm{SE}_{\mathrm{base}}^2 + \mathrm{SE}_c^2}\right)
$$

on the flop or in multi-way pots so simulation noise does not mislabel neutral cards as outs or scare cards.

In [13]:
# Exact river and heads-up turn enumeration, plus dynamic 2-sigma flop next-card scanning.
def exact_river_equity(hole, board5):
    "Compute exact river equity over all C(45, 2) = 990 opponent two-card holdings."
    dead = set(hole) | set(board5)
    rem = [c for c in FULL_DECK if c not in dead]
    v_hero = rank7fast(hole + board5)
    opp_hand = [0, 0] + list(board5)
    wins = 0.0
    total = 0
    n_rem = len(rem)
    for i in range(n_rem):
        opp_hand[0] = rem[i]
        for j in range(i + 1, n_rem):
            opp_hand[1] = rem[j]
            v_opp = rank7fast(opp_hand)
            if v_hero > v_opp:
                wins += 1.0
            elif v_hero == v_opp:
                wins += 0.5
            total += 1
    return wins / total

def next_card_scan(hole, board, num_opponents=1):
    "Evaluate next-street card sensitivity via exact turn enumeration or dynamic 2-sigma Monte Carlo."
    dead = set(hole) | set(board)
    rem = [c for c in FULL_DECK if c not in dead]

    if len(board) == 4 and num_opponents == 1:
        card_eq = {}
        for c in rem:
            card_eq[c] = (exact_river_equity(hole, board + [c]), 0.0)
        base_eq = sum(v[0] for v in card_eq.values()) / len(rem)
        base_se = 0.0
    else:
        base_res = equity(hole, board, num_opponents, iters=600)
        base_eq, base_se = base_res["equity"], base_res["se"]
        card_eq = {}
        for c in rem:
            res_c = equity(hole, board + [c], num_opponents, iters=250)
            card_eq[c] = (res_c["equity"], res_c["se"])

    helpful_cards = []
    scare_cards = []
    for c, (eq_c, se_c) in card_eq.items():
        tau = max(0.03, 2.0 * math.sqrt(base_se**2 + se_c**2))
        delta = eq_c - base_eq
        if delta > tau:
            helpful_cards.append((c, delta))
        elif delta < -tau:
            scare_cards.append((c, delta))

    return {
        "base_equity": base_eq,
        "base_se": base_se,
        "helpful_count": len(helpful_cards),
        "scare_count": len(scare_cards),
        "helpful_cards": helpful_cards,
        "scare_cards": scare_cards,
    }

### Worked equity examples and timing demo

In the cell below, we compare pocket Tens ($\text{T}\spadesuit\,\text{T}\heartsuit$) on a $\text{K}\spadesuit\,\text{Q}\diamondsuit\,5\clubsuit$ flop against a random hand, a single-raised range, and a $3$-bet range, verify $\mathrm{SE} = 0$ on a Royal Flush board lock, and run our $45{,}540$-deal exact turn scanner.

In [14]:
seeded_rng = random.Random(42)
tt_hole = parse_cards("Ts Th")
kq5_flop = parse_cards("Ks Qd 5c")

eq_uniform = equity(tt_hole, kq5_flop, num_opponents=1, iters=4000, rng=seeded_rng)
eq_1raise = equity_vs_range(tt_hole, kq5_flop, chen_floor=6.0, iters=4000, rng=seeded_rng)
eq_3bet = equity_vs_range(tt_hole, kq5_flop, chen_floor=9.0, iters=4000, rng=seeded_rng)

royal_board = parse_cards("As Ks Qs Js Ts")
chop_check = equity(parse_cards("2h 3d"), royal_board, num_opponents=1, iters=200, rng=seeded_rng)
assert chop_check["equity"] == 0.5 and chop_check["se"] == 0.0
assert is_board_lock_nut_chop(parse_cards("2h 3d"), royal_board) is True

print("Range-Conditioned Equity Benchmark: Pocket Tens (T♠ T♥) on K♠ Q♦ 5♣")
print(f"  1. vs Uniform Random Hand (Algorithm 7)         : {100*eq_uniform['equity']:.1f}% (SE = {100*eq_uniform['se']:.2f}%)")
print(f"  2. vs Single-Raise Range (Floor 6.0, Alg 8)     : {100*eq_1raise:.1f}%")
print(f"  3. vs 3-Bet Range (Floor 9.0, Alg 8)            : {100*eq_3bet:.1f}%\n")

t0 = time.perf_counter()
scan_res = next_card_scan(parse_cards("As Ks"), parse_cards("Qs Js 4h 2d"), num_opponents=1)
dt_ms = (time.perf_counter() - t0) * 1000.0
print(f"Exact Turn Enumeration (Algorithm 9: A♠ K♠ on Q♠ J♠ 4♥ 2♦ across 46 x 990 = 45,540 deals in {dt_ms:.1f} ms):")
print(f"  Exact Turn Equity : {100*scan_res['base_equity']:.2f}% (Standard Error = {scan_res['base_se']:.4f})")
print(f"  Helpful River Outs: {scan_res['helpful_count']} cards, sample: {[card_str(c) for c, _ in scan_res['helpful_cards'][:12]]}")
print(f"  Scare River Cards : {scan_res['scare_count']} cards")

Range-Conditioned Equity Benchmark: Pocket Tens (T♠ T♥) on K♠ Q♦ 5♣
  1. vs Uniform Random Hand (Algorithm 7)         : 69.4% (SE = 0.73%)
  2. vs Single-Raise Range (Floor 6.0, Alg 8)     : 54.9%
  3. vs 3-Bet Range (Floor 9.0, Alg 8)            : 22.6%

Exact Turn Enumeration (Algorithm 9: A♠ K♠ on Q♠ J♠ 4♥ 2♦ across 46 x 990 = 45,540 deals in 328.0 ms):
  Exact Turn Equity : 63.10% (Standard Error = 0.0000)
  Helpful River Outs: 18 cards, sample: ['2♠', '3♠', '4♠', '5♠', '6♠', '7♠', '8♠', '9♠', 'T♠', 'T♥', 'T♦', 'T♣']
  Scare River Cards : 28 cards


## 5. Processes IV and V: the $4$ to $8$ player betting state machine and pot settlement

### 5.1 State is the engine's memory of the hand

To prevent chip-accounting bugs across multiple betting rounds and side pots, three separate chip counters are maintained on each player $p$:

* `stack` ($p.\mathrm{stack} \ge 0$): chips still sitting in front of the player available to be wagered;
* `bet` ($p.\mathrm{bet} \ge 0$): chips wagered during the current street only (reset to $0$ at the start of the flop, turn, and river);
* `committed` ($p.\mathrm{committed} \ge 0$): cumulative chips wagered across the entire hand.

Whenever chips move from a player into the pot, all three fields and `state.pot` are updated together so that chip conservation holds at every step:

$$
\sum_{p} p.\mathrm{stack} + \mathrm{state.pot} = M_0, \qquad \mathrm{state.pot} = \sum_{p} p.\mathrm{committed}.
$$

### Data structures and atomic chip transfer

The dataclasses below represent players, pot slices, and the table state. `transfer_chips` caps any wager by the player's remaining stack and marks the player `all_in` when their stack reaches zero.

In [15]:
@dataclass
class PlayerState:
    id: int
    name: str
    stack: int
    persona: str = "SOLID"
    is_human: bool = False
    hole: List[int] = field(default_factory=list)
    bet: int = 0
    committed: int = 0
    folded: bool = False
    all_in: bool = False
    has_acted: bool = False
    acted_at_bet: int = -1
    min_raise_seen: int = 0
    busted: bool = False
    dealt_in: bool = True
    score: int = -1
    best5: Tuple[int, ...] = ()

@dataclass
class PotSlice:
    amount: int
    eligible_ids: List[int]
    contrib: List[Tuple[int, int]] = field(default_factory=list)
    level: int = 0

@dataclass
class TableState:
    players: List[PlayerState]
    sb: int = 10
    bb: int = 20
    btn_seat: int = -1
    sb_seat: int = -1
    bb_seat: int = -1
    pot: int = 0
    current_bet: int = 0
    last_raise: int = 20
    raises: int = 0
    preflop_raises: int = 0
    opened: bool = False
    aggressor_id: Optional[int] = None
    preflop_aggressor_id: Optional[int] = None
    stage: str = "preflop"
    board: List[int] = field(default_factory=list)
    deck: List[int] = field(default_factory=list)
    burned: List[int] = field(default_factory=list)
    acting_idx: int = -1

def transfer_chips(state, p, delta):
    "Move up to delta chips from player p into the pot while conserving total table chips."
    actual = max(0, min(p.stack, delta))
    p.stack -= actual
    p.bet += actual
    p.committed += actual
    state.pot += actual
    if p.stack == 0 and p.committed > 0 and not p.folded:
        p.all_in = True
    return actual

### 5.2 Starting a hand on a $4$ to $8$ player table (`start_hand`)

In our engine, starting tables are configured for **$4$, $5$, $6$, $7$, or $8$ players** (`VALID_PLAYER_COUNTS = (4, 5, 6, 7, 8)`). As eliminations occur, the active lineup contracts down to three-handed and heads-up play.

When advancing the blinds after a bustout, we follow **Poker TDA Rule 19 (Dead-Button / Forward-Moving Blinds)**: on any multi-way deal, the new small blind is placed directly at the seat that posted the big blind on the previous hand (`sb_seat = prev_bb_seat`, even if that player just busted, creating a dead small blind), and the new big blind advances clockwise to the next funded seat. This prevents a surviving player from ever skipping the big blind or paying the small blind twice in a row.

Even when the big blind player is short-stacked and all-in for less than $\mathrm{bb}$, the preflop bring-in `current_bet` is initialized to the full nominal $\mathrm{bb}$.

In [16]:
def next_funded_seat(players, from_seat):
    n = len(players)
    for k in range(1, n + 1):
        idx = (from_seat + k) % n
        if players[idx].dealt_in and players[idx].stack > 0:
            return idx
    return -1

VALID_PLAYER_COUNTS = (4, 5, 6, 7, 8)

def normalize_player_count(value):
    "Clamp the starting player count to the supported 4 to 8 player range."
    return max(4, min(8, int(value)))

def position_labels(state):
    "Assign table position badges (BTN, SB, BB, UTG, UTG+1, MP, HJ, CO) to active seats."
    n = len(state.players)
    if state.bb_seat < 0:
        return {}
    live_ids = [p.id for p in state.players if p.dealt_in]
    labels = {}
    if state.players[state.bb_seat].dealt_in:
        labels[state.bb_seat] = "BB"
    if state.players[state.sb_seat].dealt_in:
        labels[state.sb_seat] = "SB · BTN" if len(live_ids) == 2 else "SB"
    if len(live_ids) > 2 and state.players[state.btn_seat].dealt_in and state.btn_seat not in labels:
        labels[state.btn_seat] = "BTN"

    rest = []
    for i in range(1, n + 1):
        idx = (state.bb_seat + i) % n
        if state.players[idx].dealt_in and idx not in labels:
            rest.append(idx)
    pool = {
        1: ["UTG"],
        2: ["UTG", "CO"],
        3: ["UTG", "HJ", "CO"],
        4: ["UTG", "UTG+1", "HJ", "CO"],
        5: ["UTG", "UTG+1", "MP", "HJ", "CO"],
    }
    names = pool.get(len(rest), ["MP"] * len(rest))
    for idx, pos_name in zip(rest, names):
        labels[idx] = pos_name
    return labels

def start_hand(state, custom_deck=None):
    "Initialize a hand on a 4 to 8 player table using TDA Rule 19 dead-button blind advancement."
    assert 4 <= len(state.players) <= 8, f"Table must have 4 to 8 players (got {len(state.players)})"
    for p in state.players:
        p.busted = (p.stack <= 0)
        p.dealt_in = not p.busted
        p.hole = []
        p.bet = 0
        p.committed = 0
        p.folded = p.busted
        p.all_in = False
        p.has_acted = False
        p.acted_at_bet = -1
        p.min_raise_seen = state.bb
        p.score = -1
        p.best5 = ()

    eligible = [p.id for p in state.players if p.dealt_in]
    assert len(eligible) >= 2, "Need at least 2 funded players"
    n = len(state.players)
    heads = (len(eligible) == 2)

    if state.bb_seat < 0:
        if heads:
            state.btn_seat = eligible[0]
            state.sb_seat = eligible[0]
            state.bb_seat = eligible[1]
        else:
            state.btn_seat = eligible[0]
            state.sb_seat = eligible[1]
            state.bb_seat = eligible[2]
    elif heads:
        state.bb_seat = next_funded_seat(state.players, state.bb_seat)
        state.sb_seat = next_funded_seat(state.players, state.bb_seat)
        state.btn_seat = state.sb_seat
    else:
        # Under TDA Rule 19, the previous big blind seat becomes the new small blind seat,
        # and the new big blind moves clockwise to the next funded player.
        state.sb_seat = state.bb_seat
        state.bb_seat = next_funded_seat(state.players, state.sb_seat)
        state.btn_seat = (state.sb_seat - 1 + n) % n

    state.pot = 0
    state.board = []
    state.burned = []
    state.stage = "preflop"
    state.raises = 0
    state.preflop_raises = 0
    state.opened = False
    state.aggressor_id = None
    state.preflop_aggressor_id = None

    state.deck = list(custom_deck) if custom_deck is not None else shuffle_deck_in_place(list(FULL_DECK))
    for _ in range(2):
        for i in range(n):
            p = state.players[(state.sb_seat + i) % n]
            if p.dealt_in:
                p.hole.append(state.deck.pop())

    if state.players[state.sb_seat].dealt_in:
        transfer_chips(state, state.players[state.sb_seat], state.sb)
    transfer_chips(state, state.players[state.bb_seat], state.bb)

    state.current_bet = max(state.bb, state.players[state.bb_seat].bet)
    state.last_raise = state.bb
    state.acting_idx = find_next_actor(state, state.bb_seat)

### 5.3 Finding the next player who owes an action (`find_next_actor`)

A betting street remains open as long as more than one player is still in the hand and at least one non-all-in player either has not yet acted or has not yet matched `current_bet`:

$$
\neg\, p.\mathrm{hasActed} \quad \lor \quad p.\mathrm{bet} < \mathrm{currentBet}.
$$

If only one player can still act and they have already matched `current_bet` (because all other contenders are all-in), `find_next_actor` returns $-1$ so the remaining board cards can be dealt out automatically.

In [17]:
# Clockwise turn-order scanner that returns the next seat index needing action, or -1 when complete.
def find_next_actor(state, from_seat):
    "Scan clockwise for the next player who owes an action on the current street."
    live = [p for p in state.players if p.dealt_in and not p.folded]
    if len(live) <= 1:
        return -1
    can_act = [p for p in live if not p.all_in and p.stack > 0]
    if len(can_act) == 0:
        return -1
    if len(can_act) == 1 and can_act[0].bet >= state.current_bet:
        return -1
    n = len(state.players)
    for k in range(1, n + 1):
        idx = (from_seat + k) % n
        p = state.players[idx]
        if not p.dealt_in or p.folded or p.all_in or p.stack <= 0:
            continue
        if (not p.has_acted) or (p.bet < state.current_bet):
            return idx
    return -1

### 5.4 Legal raises and cumulative short all-ins (`can_reopen` and `apply_action`)

In No-Limit Hold'em, a full raise must increase `current_bet` by at least `last_raise`:

$$
\tau_{\min} = \max(\mathrm{currentBet} + \mathrm{lastRaise},\, \mathrm{bb}),
$$

capped by the player's available total $\tau_{\max} = p.\mathrm{bet} + p.\mathrm{stack}$. When a short stack goes all-in for less than $\tau_{\min}$, that short all-in does not by itself reopen raising for a player who has already acted. However, if multiple short all-ins stack up so that the cumulative increase since `p.acted_at_bet` satisfies

$$
\mathrm{currentBet} - p.\mathrm{actedAtBet} \ge \mathrm{lastRaise},
$$

raising is legally reopened for player $p$.

In [18]:
# Here we check whether a player has raising rights and apply Fold, Check, Call, or Raise transitions.
def can_reopen(state, p):
    "Return True if player p is permitted to raise, accounting for cumulative short all-ins."
    to_call = max(0, state.current_bet - p.bet)
    if p.stack <= to_call:
        return False
    if p.acted_at_bet < 0:
        return True
    return (state.current_bet - p.acted_at_bet) >= state.last_raise

def clamp_raise(state, p, desired_to):
    "Round a proposed bet or raise target to natural chip steps and clamp to legal bounds."
    max_to = p.bet + p.stack
    min_to = min(max_to, max(state.current_bet + state.last_raise, state.bb))
    step = 25 if state.bb >= 50 else (5 if state.bb >= 20 else 1)
    rounded = int(round(desired_to / step) * step)
    return min(max_to, max(min_to, rounded))

def apply_action(state, p, action, target_to=0):
    "Execute Fold, Check, Call, or Raise for player p and advance acting_idx."
    if state.stage == "preflop" and action in ("call", "raise"):
        state.opened = True
    to_call = max(0, state.current_bet - p.bet)
    if action == "fold":
        p.folded = True
        p.has_acted = True
    elif action == "check":
        assert to_call == 0, "Cannot check when facing a bet"
        p.has_acted = True
    elif action == "call":
        transfer_chips(state, p, to_call)
        p.has_acted = True
    elif action == "raise":
        assert can_reopen(state, p), f"Player {p.name} is not permitted to reopen raising"
        max_to = p.bet + p.stack
        min_to = min(max_to, max(state.current_bet + state.last_raise, state.bb))
        tau = min(max_to, max(min_to, target_to))
        delta = tau - p.bet
        prev_bet = state.current_bet
        raise_inc = tau - prev_bet
        transfer_chips(state, p, delta)
        if tau > prev_bet:
            if raise_inc >= state.last_raise:
                state.last_raise = raise_inc
            state.raises += 1
            if state.stage == "preflop":
                state.preflop_raises += 1
                state.preflop_aggressor_id = p.id
            state.aggressor_id = p.id
            for q in state.players:
                if q.id != p.id and q.dealt_in and not q.folded and not q.all_in:
                    q.has_acted = False
            state.current_bet = tau
        p.has_acted = True
    else:
        raise ValueError(f"Unknown action: {action}")

    p.acted_at_bet = state.current_bet
    p.min_raise_seen = state.last_raise
    state.acting_idx = find_next_actor(state, p.id)

### 5.5 Returning uncalled excess (`refund_street_uncalled` and `refund_uncalled`)

Whenever the highest contribution $c_1$ exceeds the second-highest contribution $c_2$ (because everyone else folded or called all-in for less), the unmatched excess

$$
\Delta_{\mathrm{uncalled}} = \max(0,\, c_1 - c_2)
$$

is not part of any pot and is returned immediately to the player who wagered it. We provide both `refund_street_uncalled` (for the end of a betting street) and `refund_uncalled` (for end-of-hand settlement).

In [19]:
# Unmatched excess chips are returned to the sole highest bettor or committer.
def refund_street_uncalled(state):
    "Return unmatched street bet excess at the end of a betting round."
    in_hand = sorted([p for p in state.players if p.dealt_in and p.bet > 0], key=lambda x: x.bet, reverse=True)
    if not in_hand:
        return 0
    top = in_hand[0]
    second = in_hand[1].bet if len(in_hand) >= 2 else 0
    if top.bet > second:
        excess = top.bet - second
        top.bet -= excess
        top.committed -= excess
        top.stack += excess
        state.pot -= excess
        if top.stack > 0:
            top.all_in = False
        return excess
    return 0

def refund_uncalled(state):
    "Return unmatched total commitment excess before pot slicing."
    committers = sorted([p for p in state.players if p.dealt_in and p.committed > 0], key=lambda x: x.committed, reverse=True)
    if not committers:
        return 0
    top = committers[0]
    second = committers[1].committed if len(committers) >= 2 else 0
    if top.committed > second:
        excess = top.committed - second
        top.committed -= excess
        top.bet = max(0, top.bet - excess)
        top.stack += excess
        state.pot -= excess
        if top.stack > 0:
            top.all_in = False
        return excess
    return 0

### 5.6 Main pots and side pots by horizontal contribution bands (`compute_pots`)

When players go all-in for unequal amounts, the pot is partitioned into horizontal commitment bands defined by the sorted distinct positive commitment levels:

$$
0 = L_0 < L_1 < L_2 < \cdots < L_k.
$$

Each player $p$ with total commitment $c_p$ contributes

$$
\gamma(p, L_{j-1}, L_j) = \max\!\big(0,\, \min(c_p, L_j) - \min(c_p, L_{j-1})\big)
$$

to band $(L_{j-1}, L_j]$. A player is eligible to win band $j$ if and only if they have not folded and $c_p \ge L_j$. Adjacent bands with identical eligibility lists are merged, and each slice records its per-player contributions `contrib` so that if a slice ever has no eligible claimants (`eligible_ids == []`), its chips are refunded directly to the contributors.

In [20]:
# Horizontal side-pot slicing across sorted distinct commitment levels.
def compute_pots(state):
    "Slice player commitments into horizontal main and side pots and merge identical eligibility bands."
    parts = [p for p in state.players if p.dealt_in and p.committed > 0]
    levels = sorted({p.committed for p in parts})
    raw_pots = []
    prev = 0

    for lvl in levels:
        band_amount = 0
        contrib = []
        for p in parts:
            c = max(0, min(p.committed, lvl) - min(p.committed, prev))
            if c > 0:
                contrib.append((p.id, c))
                band_amount += c
        eligible = [p.id for p in parts if not p.folded and p.committed >= lvl]
        if band_amount > 0:
            raw_pots.append(PotSlice(amount=band_amount, eligible_ids=eligible, contrib=contrib, level=lvl))
        prev = lvl

    merged = []
    for pot in raw_pots:
        if merged and merged[-1].eligible_ids == pot.eligible_ids:
            merged[-1].amount += pot.amount
            merged[-1].contrib.extend(pot.contrib)
            merged[-1].level = pot.level
        else:
            merged.append(pot)
    return merged

### 5.7 Showdown evaluation and odd-chip distribution (`award_pots`)

At showdown, each un-folded player is scored via `best7`. For each pot slice of amount $A$ with $m$ tied winners, every tied winner receives

$$
\mathrm{share} = \left\lfloor \frac{A}{m} \right\rfloor, \qquad \mathrm{remainder} = A \bmod m,
$$

and the $\mathrm{remainder}$ indivisible odd chips ($1$ chip each) are awarded one by one in clockwise seat order starting immediately left of the dealer button (Poker TDA Rule 20).

In [21]:
# Evaluate showdown hands, settle each pot slice, and distribute odd chips clockwise left of the button.
def award_pots(state):
    "Settle main and side pots and award odd chips clockwise from left of the button."
    refunded = refund_uncalled(state)
    live = [p for p in state.players if p.dealt_in and not p.folded]
    results = []

    if len(live) > 1:
        for p in live:
            p.score, p.best5 = best7(p.hole + state.board)

    pots = compute_pots(state)
    n_seats = len(state.players)
    for idx, ps in enumerate(pots):
        if len(ps.eligible_ids) == 0:
            for pid, amt in ps.contrib:
                state.players[pid].stack += amt
                refunded += amt
            continue

        if len(ps.eligible_ids) == 1 or len(live) == 1:
            winners = [state.players[ps.eligible_ids[0]]]
        else:
            elig_players = [state.players[i] for i in ps.eligible_ids]
            best_sc = max(p.score for p in elig_players)
            winners = [p for p in elig_players if p.score == best_sc]

        winners.sort(key=lambda w: (w.id - state.btn_seat - 1 + n_seats) % n_seats)

        share = ps.amount // len(winners)
        rem = ps.amount % len(winners)
        payouts = []
        for k, w in enumerate(winners):
            gain = share + (1 if k < rem else 0)
            w.stack += gain
            payouts.append((w.id, gain))
        results.append({"pot_index": idx, "amount": ps.amount, "winners": payouts, "refunded": refunded})

    state.pot = 0
    return results

### Worked side-pot slicing example

In the cell below, four players commit $100$, $200$, $350$, and $350$ chips, where Seat 1 (who committed $200$) has folded. Our horizontal slicer produces a $400$-chip Main Pot eligible to Seats `[0, 2, 3]` and merges the upper two bands into a $600$-chip Side Pot 1 eligible to Seats `[2, 3]`.

In [22]:
demo_players = [
    PlayerState(0, "Player A", stack=0, committed=100, all_in=True),
    PlayerState(1, "Player B", stack=200, committed=200, folded=True),
    PlayerState(2, "Player C", stack=0, committed=350, all_in=True),
    PlayerState(3, "Player D", stack=650, committed=350),
]
demo_state = TableState(players=demo_players, pot=1000)
sliced_pots = compute_pots(demo_state)
print("Worked 4-Player Multi-Way Side-Pot Decomposition (Algorithm 14):")
for i, sl in enumerate(sliced_pots):
    label = "Main Pot" if i == 0 else f"Side Pot {i}"
    print(f"  {label:<10}: Amount = {sl.amount:>4d} chips | Eligible Seat IDs = {sl.eligible_ids}")
assert [(p.amount, p.eligible_ids) for p in sliced_pots] == [(400, [0, 2, 3]), (600, [2, 3])]

Worked 4-Player Multi-Way Side-Pot Decomposition (Algorithm 14):
  Main Pot  : Amount =  400 chips | Eligible Seat IDs = [0, 2, 3]
  Side Pot 1: Amount =  600 chips | Eligible Seat IDs = [2, 3]


## 6. Process VI and Process VII: AI policy, game theory, tournament ICM, and opponent adaptation

### 6.1 Player personas and difficulty interpolation

Each computer opponent is given a persona vector $(\theta, \alpha, \beta)$ of tightness, aggression, and bluff propensity. As the difficulty setting increases from `casual` ($\lambda = 0$) to `solid` ($\lambda = 0.35$) to `sharp` ($\lambda = 0.70$), each persona's parameters are linearly interpolated toward our `SOLID` baseline:

$$
\theta_{\mathrm{eff}} = (1 - \lambda)\theta_{\mathrm{persona}} + \lambda\theta_{\mathrm{solid}}.
$$

In [23]:
PERSONAS = {
    "TAG":   {"tight": 0.05, "aggr": 0.62, "bluff": 0.13},
    "LAG":   {"tight": -0.07, "aggr": 0.80, "bluff": 0.30},
    "CALL":  {"tight": -0.10, "aggr": 0.20, "bluff": 0.09},
    "NIT":   {"tight": 0.17, "aggr": 0.30, "bluff": 0.03},
    "SOLID": {"tight": 0.05, "aggr": 0.55, "bluff": 0.12},
}
DIFF_PULL = {"casual": 0.00, "solid": 0.35, "sharp": 0.70}

def effective_persona(code, difficulty="solid"):
    "Interpolate a persona's parameters toward the SOLID baseline according to difficulty."
    base = PERSONAS.get(code, PERSONAS["SOLID"])
    solid = PERSONAS["SOLID"]
    lam = DIFF_PULL.get(difficulty, 0.35)
    return {
        "tight": (1.0 - lam) * base["tight"] + lam * solid["tight"],
        "aggr":  (1.0 - lam) * base["aggr"]  + lam * solid["aggr"],
        "bluff": (1.0 - lam) * base["bluff"] + lam * solid["bluff"],
    }

### 6.2 $169$ starting-hand classes, our embedded `PF_TABLE` Nash equilibrium lookup, and CFR

By suit isomorphism, the $\binom{52}{2} = 1{,}326$ two-card starting combinations compress into $169$ strategic classes on a $13 \times 13$ grid: $13$ pocket pairs on the diagonal ($6$ combos each), $78$ suited hands above the diagonal ($4$ combos each), and $78$ offsuit hands below the diagonal ($12$ combos each):

$$
13 \times 6 + 78 \times 4 + 78 \times 12 = 78 + 312 + 936 = 1{,}326.
$$

In the cell below, we include **both**:
1. Our exact $6{,}422$-nybble precomputed $169 \times 19$ Heads-Up Nash Equilibrium Push/Fold Table (`PF_TABLE` and `pf_lookup`), which stores equilibrium shoving and calling probabilities across stack depths $2\text{ bb}$ through $20\text{ bb}$.
2. A self-contained Counterfactual Regret Minimization (`solve_push_fold_cfr_demo`) routine that illustrates how regret matching iteratively updates strategy probabilities.

In [24]:
PF_MIN = 2
PF_MAX = 20
PF_TABLE = "fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff1ffffff102000fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff00fffffffffff00ffffffffffff0fffffff4fff00ffffff1010f00fffff000000f0fffff0000000fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff93fff0fffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff00ffffffffffe00fffffffffff00ffffffffffff0fffff100bff00fffff00000f00ffff0000000f0ffff00000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff00fffffffffff10ffffffffffff2ffffff1cffff0fffff100ffff0fffff000000f0ffff30000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffe00ffffffffffa00ffffffffffe00fffffffffffe0ffffb000dff00ffffe00000f40ffff0000000f0ffff00000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff100fffffffff1000ffffff9fff000ffffff11ff110ffff10000ff00ffff100000f00fffc0000000f0fff100000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffe00fffffffffe000ffffffefffe00fffffeeeffff0ffff0000effd0ffff000003fd0fff00000000f0fff000000000fffffffffffffffffffffffffffffffffffffffffffffffffffcefffffffffde00fffffffff0000fffffffef0000fffff00ff0000ffffe000f0000fff100000f000fff0000000f00ff100000000f0ff1000000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffeefffffffffea00fffffffffe100fffffe1fffe00fffff2defffe0fff30000effb0fffa000007fe0fff00000000f0fff000000000ffffffffffffffffffffffffffffffffffffffffffffffffffe00fffffffff0000fffffffe00000ffffe0f000000ffff000f30000fffd0000f0000fff000000f000ff60000000f00ff000000000f0ff0000000000ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffdefffffffffed00fffffffffe000fffffe0ffee00fffff0defffe0fff00000effd0fff000000dfe0ff000000000f0ff0000000000ffffffffffffffffffffffffffffffffffffff10ffffffffe0000fffffffdd0000fffffff000000ffff00f000000fff0000f00000fff00000f0000ff0000000f000ff00000000f00ff000000000f0fe0000000000fffffffffffffffffffffffffffffffffffffffffffffffffffeffffffffffff18fffffffffe100fffffffffe000ffffe20ffee00fffbf02effee0fff00000effc0ffd0000005fe0ff000000000f0ff0000000000fffffffffffffffffffffffffffffffffffff200fffffffe00000fffffff000000ffffef1000000fffe00f000000ff50000f00000ff800000f0000ff0000000f000ff00000000f00f0000000000f0f00000000000fffffffffffffffffffffffffffffffffffffffffffffffffffeefffffffffef01fffffffffe100fffffffffe000fff3d20ffee00fff2e06effee0ff100000effb0ff10000008fe0ff000000000f0ff0000000000ffffffffffffffffffffffffffffffffffff0000fffffff200000ffffffd000000ffff9f0000000ffe000f000000ff00000f00000ff000000f0000ff0000000f000f400000000f00f0000000000f0f00000000000fffffffffffffffffffffffffffffffffffffffffffffffffff4efffffffffef00fffffffffe100ffffeffffe100ff41410ffee00fff0e02effee0ff000000effa0ff00000008fe0ff000000000f0ff0000000000fffffffffffffffffffffffff30fffffffde0000ffffffe000000fffffe1000000fffd0f0000000ff0000f000000ff00000f00000ff000000f0000f50000000f000f000000000f00f0000000000f0f00000000000fffffffffffffffffffffffffffffffffffffff0fffffffffff03fffffffffee00fffffffffe000ffff9efffe000ff00000ffee00ff20900effee0ff000000dfe00ff00000000fd0fb000000000f0f20000000000fffffffffffffffffffffffff00ffffffe000000ffffff8000000fffff90000000ffe00f0000000ff0000f000000ff00000f00000fe000000f0000f00000000f000f000000000f00f0000000000f0f00000000000ffffffffffffffffffffffffffffffffffffffe1ffffffffffe00fffffffffee00fffffffffe000ff5f1efffd000ff00010ffee00ff30704effee0ff000000dfe00ff00000004fa0f1000000000f0f00000000000fffffffffffffffffffffffc100ffffff6000000ffffff0000000fffff00000000ff700f0000000ff0000f000000f300000f00000f3000000f0000f00000000f000f000000000f00f0000000000f0f00000000000ffffffffffffffffffffffffffffffffffffffe0fffffffffee03fffffffffee00ffffffffee000ff4f1efefe000ff00000ffed00ff10101efeee0ff000000dfe00ff00000000fc0f1000000000f0f00000000000fffffffffffffffffffffff0000ffffff0000000fffffd0000000fffef00000000ff000f0000000fd0000f000000f000000f00000f0000000f0000f00000000f000f000000000f00e0000000000f0100000000000ffffffffffffffffffffffffffffffffffffffe0fffffffffee00fffffffffde00ffffffffed000ff0e1efefd000ff00000ffed00ff00000dfeee0ff000000cfe00ff00000000f90f0000000000f0f00000000000fffffffffffffffffffffff0000ffffff0000000fffffb0000000fffdf00000000ff000f0000000f00000f000000f000000f00000f0000000f0000f00000000f000f000000000f0000000000000f0000000000000fffffffffffffffffffffffffffffffffffffe70ffffffffeed00fffffffff4e00ffffffffe3000ff0e0efee7000ff00000ffe300ff000009feee0f20000000fe00f000000000f00f0000000000f0f00000000000fffffffffffffffffffffbb0000fffffd0000000fffff00000000ffe9f00000000ff000f0000000f00000f000000f000000f00000f0000000f0000f00000000f000f000000000f0000000000000f0000000000000ffffffffffffffffffffffffffffffffffffee20ffffffffeed00fffffffef1e00ff9feffee0000f40e0efee1000f800000ffd100ff000004feee0f00000000fe00f000000000f00f0000000000f0f00000000000ffffffffffffffffffffd000000fffff90000000ffffe00000000ffc0f00000000ff000f0000000f00000f000000f000000f00000f0000000f0000f00000000f0009000000000f0000000000000f0000000000000ffffffffffffffffffffffffffffffffffffee00fffffffeeed00fffffffee1d00ffaeeffee0000f30e0dfee0000f400000fed300ff000000feed0f00000000fe00f000000000f00f0000000000f0f00000000000fffffffffffff5ffffffa000000fffff00000000ffffd00000000ff50f00000000ff000f0000000f00000f000000f000000f00000f0000000f0000d00000000f0001000000000f0000000000000f0000000000000fffffffffffffffffffffffffeefffffffefee00fffffffeedd00fffffffee2d00ff8eeffee2000f10a0dfee9000f200000fed000f7000000fedd0f00000000fe00f000000000f00f0000000000f0f00000000000ffffffffffffe1ffffff0000000fffff00000000ffffc00000000ff00f00000000fd000f0000000f00000f000000f000000f00000f0000000f0000300000000f0000000000000f0000000000000f0000000000000fffffffffffffffffffffffffeefffffffefee00fffffffeeed00fffffffee0d00fffeeffee5000f1030dfee1000f000000fed000f6000000fedd0f00000000fe00f000000000f00f0000000000f0f00000000000ffffffffffffc0ffffff0000000ffffe00000000ffefc00000000ff00f00000000f0000f0000000f00000f000000f000000f00000e0000000f0000100000000f0000000000000f0000000000000f0000000000000f"

def hand_to_169_index(c1, c2):
    "Map any 2 hole cards (c1, c2) to their canonical 13x13 grid index in 0..168."
    r1, r2 = c1 >> 2, c2 >> 2
    s1, s2 = c1 & 3, c2 & 3
    hi, lo = 12 - max(r1, r2), 12 - min(r1, r2)
    if r1 == r2 or s1 == s2:
        row, col = hi, lo
    else:
        row, col = lo, hi
    return row * 13 + col

def pf_lookup(cls_idx, eff_bb, which):
    "Look up the equilibrium shove (which=0) or call (which=1) probability from PF_TABLE."
    d = max(PF_MIN, min(PF_MAX, int(round(eff_bb))))
    offset = (d - PF_MIN) * 338 + which * 169 + cls_idx
    return int(PF_TABLE[offset], 16) / 15.0

def pf_table_combo_summary(eff_bb):
    "Compute the exact combo-weighted SB shove share and BB call share from PF_TABLE."
    shove_combos = 0.0
    call_combos = 0.0
    for row in range(13):
        for col in range(13):
            cls_idx = row * 13 + col
            w = 6.0 if row == col else (4.0 if row < col else 12.0)
            shove_combos += w * pf_lookup(cls_idx, eff_bb, 0)
            call_combos += w * pf_lookup(cls_idx, eff_bb, 1)
    return shove_combos / 1326.0, call_combos / 1326.0

def solve_push_fold_cfr_demo(stack_bb=10.0, iterations=800):
    "Illustrate Counterfactual Regret Minimization (regret matching) on a 169-class push/fold game."
    weights = [0.0] * 169
    chen_169 = [0.0] * 169
    for row in range(13):
        for col in range(13):
            idx = row * 13 + col
            r_hi, r_lo = 12 - min(row, col), 12 - max(row, col)
            if row == col:
                weights[idx] = 6.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 1))
            elif row < col:
                weights[idx] = 4.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 0))
            else:
                weights[idx] = 12.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 1))

    eq_matrix = [[0.5] * 169 for _ in range(169)]
    for i in range(169):
        for j in range(i + 1, 169):
            p_ij = 1.0 / (1.0 + math.exp(-0.18 * (chen_169[i] - chen_169[j])))
            eq_matrix[i][j] = p_ij
            eq_matrix[j][i] = 1.0 - p_ij

    regret_sb = [0.0] * 169
    regret_bb = [0.0] * 169
    sum_push = [0.0] * 169
    sum_call = [0.0] * 169

    for t in range(1, iterations + 1):
        s_push = [1.0 if regret_sb[i] > 0 else 0.0 for i in range(169)]
        c_call = [1.0 if regret_bb[j] > 0 else 0.0 for j in range(169)]
        if t == 1:
            s_push = [0.5] * 169
            c_call = [0.5] * 169

        for i in range(169):
            sum_push[i] += s_push[i]
            sum_call[i] += c_call[i]

        for i in range(169):
            ev_push = 0.0
            w_sum = 0.0
            for j in range(169):
                w = weights[j]
                w_sum += w
                ev_push += w * ((1.0 - c_call[j]) * 1.0 + c_call[j] * stack_bb * (2.0 * eq_matrix[i][j] - 1.0))
            ev_push /= w_sum
            regret_sb[i] = max(0.0, regret_sb[i] + (ev_push - (-0.5)))

        for j in range(169):
            ev_call = 0.0
            w_sum = 0.0
            for i in range(169):
                w = weights[i] * s_push[i]
                w_sum += w
                ev_call += w * ((1.0 - eq_matrix[i][j]) * (2.0 * stack_bb) - (stack_bb - 1.0))
            if w_sum > 0:
                ev_call /= w_sum
                regret_bb[j] = max(0.0, regret_bb[j] + (ev_call - (-1.0)))

    avg_push = [sum_push[i] / iterations for i in range(169)]
    avg_call = [sum_call[j] / iterations for j in range(169)]
    combo_push_pct = sum(weights[i] * avg_push[i] for i in range(169)) / 1326.0
    combo_call_pct = sum(weights[j] * avg_call[j] for j in range(169)) / 1326.0
    return combo_push_pct, combo_call_pct

### 6.3 Bayesian opponent reads (`bayesian_stat`)

We track five binary tendencies on the human player (`vpip`, `pfr`, `foldToRaise`, `foldToBet`, and `aggr`). Each rate is modeled with a conjugate Beta prior $\mathrm{Beta}(a_0, b_0)$. After $k$ occurrences in $n_{\mathrm{obs}}$ opportunities, the posterior distribution is

$$
p \mid \text{data} \sim \mathrm{Beta}(a_0 + k,\, b_0 + n_{\mathrm{obs}} - k),
$$

with posterior mean and posterior standard deviation

$$
\mu = \frac{a}{a + b}, \qquad \sigma = \sqrt{\frac{ab}{(a + b)^2(a + b + 1)}}.
$$

To avoid reacting to small-sample noise, our exploitation signal $z \in [-1, 1]$ remains zero until $|\mu - \mu_0| > 2\sigma$.

In [25]:
PRIORS = {
    "vpip":        (5.0, 7.0),
    "pfr":         (3.0, 11.0),
    "foldToRaise": (6.0, 6.0),
    "foldToBet":   (6.0, 6.0),
    "aggr":        (4.0, 10.0),
}

def bayesian_stat(key, k_succ, n_obs):
    "Compute the Beta-Binomial posterior mean, standard deviation, and 2-sigma gated signal z."
    a0, b0 = PRIORS[key]
    a = a0 + k_succ
    b = b0 + (n_obs - k_succ)
    n = a + b
    mu0 = a0 / (a0 + b0)
    mu = a / n
    sigma = math.sqrt((a * b) / (n * n * (n + 1.0)))
    delta = mu - mu0
    z = max(-1.0, min(1.0, delta / (4.0 * sigma))) if abs(delta) > 2.0 * sigma else 0.0
    return {"mean": mu, "prior_mean": mu0, "sigma": sigma, "z": z, "n_obs": n_obs}

### 6.4 Malmuth-Harville tournament ICM and postflop decision policy

In tournament play, busting forfeits all future pay jumps, so chips lost are worth more in prize equity than chips won. Under the Malmuth-Harville Independent Chip Model (`compute_icm`), the probability that player $i$ finishes first is $P(\text{1st} = i) = S_i / \sum_k S_k$, and subsequent places are computed recursively over the remaining players. Comparing the prize-equity loss of busting against the prize-equity gain of doubling up defines the **Bubble Factor** $\mathrm{BF} \ge 1$ (`compute_bubble_metrics`), which raises the break-even calling equity to:

$$
q^*_{\mathrm{ICM}} = \frac{\mathrm{BF} \cdot C}{P + \mathrm{BF} \cdot C}.
$$

Our postflop policy (`ai_postflop_decision`) then combines:
* multi-way range conditioning and preflop card-removal bunching (`equity_multiway_ranged`);
* river board-lock chop protection (`is_board_lock_nut_chop`);
* a square-root bet-pressure discount:

$$
\hat{e} = e - \min\!\left(c_{\max},\, \gamma\sqrt{\mathrm{pressure}}\right) - 0.025(m - 1), \qquad \mathrm{pressure} = \frac{\mathrm{toCall}}{\max(\mathrm{bb},\, \mathrm{pot})};
$$

* and the tournament ICM survival premium when calling a large fraction of the stack.

In [26]:
def compute_icm(stacks, payouts):
    "Compute recursive Malmuth-Harville Independent Chip Model tournament prize equities."
    n = len(stacks)
    total = sum(stacks)
    if total <= 0:
        return [0.0] * n
    ev = [0.0] * n

    def recurse(mask, rem_total, depth, prob_acc):
        if depth >= len(payouts) or prob_acc < 1e-9:
            return
        prize = payouts[depth]
        for i in range(n):
            if (mask & (1 << i)) or stacks[i] <= 0:
                continue
            p_win = stacks[i] / rem_total
            branch_prob = prob_acc * p_win
            ev[i] += branch_prob * prize
            if depth + 1 < len(payouts) and rem_total - stacks[i] > 0:
                recurse(mask | (1 << i), rem_total - stacks[i], depth + 1, branch_prob)

    recurse(0, float(total), 0, 1.0)
    return ev

def compute_bubble_metrics(stacks, hero_idx, opp_idx, to_call, pot, payouts):
    "Compute hero's tournament ICM equity, Bubble Factor, and survival equity premium."
    base_icm = compute_icm(stacks, payouts)
    fold_s = list(stacks)
    fold_s[opp_idx] += pot
    win_s = list(stacks)
    win_s[hero_idx] += pot
    lose_s = list(stacks)
    lose_s[hero_idx] = max(0.0, lose_s[hero_idx] - to_call)
    lose_s[opp_idx] += pot + to_call

    v_fold = compute_icm(fold_s, payouts)[hero_idx]
    v_win = compute_icm(win_s, payouts)[hero_idx]
    v_lose = compute_icm(lose_s, payouts)[hero_idx]
    risk = max(0.0, v_fold - v_lose)
    gain = max(1e-6, v_win - v_fold)
    chip_odds = to_call / max(1.0, pot)
    bf = max(1.0, min(2.6, (risk / gain) / max(0.05, chip_odds)))
    cev_need = to_call / (pot + to_call)
    icm_need = (bf * to_call) / (pot + bf * to_call)
    return {
        "icm_ev": base_icm[hero_idx],
        "bubble_factor": bf,
        "cev_need": cev_need,
        "icm_need": icm_need,
        "icm_premium": max(0.0, icm_need - cev_need),
    }

def opponent_chen_floor(state, opp, opp_reads):
    "Compute an action-conditioned Chen floor in [2.0, 10.5] for opponent opp."
    raises = state.preflop_raises
    floor = 8.5 if raises >= 2 else (5.5 if raises == 1 else 2.5)
    per = effective_persona(opp.persona, "solid")
    floor += per["tight"] * 12.0
    if state.preflop_aggressor_id == opp.id:
        floor += 1.0
    if state.aggressor_id == opp.id and state.raises >= 2:
        floor += 0.8
    if opp.is_human and opp_reads:
        if state.preflop_aggressor_id == opp.id and "pfr" in opp_reads and opp_reads["pfr"]["n_obs"] >= 6:
            floor += max(-1.2, min(1.2, (0.22 - opp_reads["pfr"]["mean"]) * 6.0))
        elif "vpip" in opp_reads and opp_reads["vpip"]["n_obs"] >= 6:
            floor += max(-1.2, min(1.2, (0.42 - opp_reads["vpip"]["mean"]) * 4.0))
    return max(2.0, min(10.5, floor))

def ai_postflop_decision(state, p, opp_reads=None, icm_payouts=None):
    "Choose a postflop action using multi-way range equity, board-lock chop protection, and ICM."
    opp_reads = opp_reads or {}
    to_call = min(p.stack, max(0, state.current_bet - p.bet))
    pot_odds = to_call / (state.pot + to_call) if to_call > 0 else 0.0
    live_opps = [q for q in state.players if q.dealt_in and not q.folded and q.id != p.id]
    m = max(1, len(live_opps))
    folded_cnt = sum(1 for q in state.players if q.dealt_in and q.folded)
    per = effective_persona(p.persona, "solid")

    opp_floors = [
        min(10.5, opponent_chen_floor(state, o, opp_reads) + (0.6 if to_call > 0 and state.aggressor_id == o.id else 0.0))
        for o in live_opps
    ]
    mw_stats = equity_multiway_ranged(p.hole, state.board, opp_floors, folded_count=folded_cnt, iters=900)
    eq_val = mw_stats["equity"]
    board_locked_chop = (len(state.board) == 5 and is_board_lock_nut_chop(p.hole, state.board)) or (
        len(state.board) == 5 and mw_stats["loss_prob"] == 0.0 and mw_stats["tie_prob"] > 0.98
    )

    can_r = can_reopen(state, p)
    if board_locked_chop:
        if to_call == 0:
            if can_r:
                return "raise", clamp_raise(state, p, p.bet + 0.75 * state.pot), {"raw_eq": eq_val, "board_lock": True}
            return "check", 0, {"raw_eq": eq_val, "board_lock": True}
        return "call", 0, {"raw_eq": eq_val, "board_lock": True}

    pressure = to_call / max(state.bb, state.pot)
    gamma_p, cap_p = (0.06, 0.10)
    adj_eq = eq_val - min(cap_p, gamma_p * math.sqrt(pressure)) - 0.025 * (m - 1)

    if to_call == 0:
        if eq_val > 0.62 and can_r:
            target = clamp_raise(state, p, p.bet + 0.66 * state.pot)
            return "raise", target, {"raw_eq": eq_val, "adj_eq": adj_eq}
        return "check", 0, {"raw_eq": eq_val, "adj_eq": adj_eq}

    icm_premium = 0.0
    if icm_payouts and to_call >= 0.25 * p.stack:
        stacks_now = [float(q.stack) for q in state.players if q.dealt_in]
        alive_ids = [q.id for q in state.players if q.dealt_in]
        if p.id in alive_ids:
            h_idx = alive_ids.index(p.id)
            o_id = state.aggressor_id if state.aggressor_id in alive_ids else live_opps[0].id
            o_idx = alive_ids.index(o_id)
            bm = compute_bubble_metrics(stacks_now, h_idx, o_idx, float(to_call), float(state.pot), icm_payouts)
            icm_premium = min(0.14, bm["icm_premium"] * 0.65)

    need = pot_odds + per["tight"] + (0.07 if state.raises >= 3 else 0.0) + (0.05 if to_call >= p.stack else 0.0) + icm_premium
    if eq_val > 0.74 and can_r:
        target = clamp_raise(state, p, state.current_bet + 0.75 * (state.pot + to_call))
        return "raise", target, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}
    if adj_eq >= need:
        return "call", 0, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}
    return "fold", 0, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}

### 6.5 Post-hand self-review with `equity_vs_known` (`review_hand_trace`)

After a showdown, we score past calling decisions by their expected value at the moment of the decision rather than by the realized river outcome:

$$
\mathrm{EV}_{\mathrm{call}} = q_{\mathrm{true}} \cdot P - (1 - q_{\mathrm{true}}) \cdot C.
$$

Crucially, `equity_vs_known` computes $q_{\mathrm{true}}$ conditioned on the exact hole cards revealed by showdown participants (`known_opps`), while any opponent who folded or mucked before revealing (`unknown_mucked`) is re-sampled uniformly from the unseen deck so unrevealed cards are never peeked at.

In [27]:
# Counterfactual equity against revealed showdown hands, keeping mucked hands hidden.
def equity_vs_known(hole, known_opps, board, unknown_mucked=0, iters=240, rng=None):
    "Compute equity conditioned on known showdown hole cards while re-sampling mucked opponents."
    if rng is None:
        rng = random
    dead = set(hole) | set(board)
    for ko in known_opps:
        dead.update(ko)
    stub = [c for c in FULL_DECK if c not in dead]
    L = len(stub)
    need = 5 - len(board)
    draw = need + 2 * unknown_mucked

    def score_runout(full_board, extra_opps):
        mine = rank7fast(hole + full_board)
        ties = 0
        for o in known_opps:
            r = rank7fast(o + full_board)
            if r > mine:
                return 0.0
            if r == mine:
                ties += 1
        for o in extra_opps:
            r = rank7fast(o + full_board)
            if r > mine:
                return 0.0
            if r == mine:
                ties += 1
        return 1.0 / (1 + ties)

    if draw == 0:
        return score_runout(list(board), [])

    total = 0.0
    for _ in range(iters):
        for k in range(draw):
            j = k + int(rng.random() * (L - k))
            stub[k], stub[j] = stub[j], stub[k]
        fb = list(board) + stub[:need]
        extra = [[stub[need + 2 * u], stub[need + 2 * u + 1]] for u in range(unknown_mucked)]
        total += score_runout(fb, extra)
    return total / iters

def review_hand_trace(trace, shown_hands, learn_store, bb=20):
    "Update post-decision EV regret buckets using only showdown-revealed hole cards."
    for d in trace:
        if d["to_call"] <= 0:
            continue
        known_opps = [shown_hands[oid] for oid in d["opp_ids"] if oid in shown_hands]
        unknown_mucked = sum(1 for oid in d["opp_ids"] if oid not in shown_hands)
        q_true = equity_vs_known(d["hole"], known_opps, d["board"], unknown_mucked=unknown_mucked, iters=240, rng=seeded_rng)
        d["true_eq"] = q_true
        ev_fold = 0.0
        ev_call = q_true * d["pot"] - (1.0 - q_true) * d["to_call"]
        base_ev = ev_fold if d["action"] == "fold" else ev_call
        bucket = learn_store.setdefault(d["key"], [0.0, 0.0, 0])
        bucket[0] += (ev_fold - base_ev) / bb
        bucket[1] += (ev_call - base_ev) / bb
        bucket[2] += 1

### Demonstrations: `PF_TABLE` equilibrium summary, CFR demo, ICM bubble factor, and Bayesian gating

In the cell below, we compare our embedded `PF_TABLE` Nash equilibrium frequencies against the simplified CFR demo, evaluate a $4$-player tournament ICM bubble spot, and test the $2\sigma$ Bayesian gate and hindsight-free review.

In [28]:
print("Exact Embedded PF_TABLE Nash Equilibrium Summary vs Simplified CFR Demo:")
for depth in [20, 15, 10, 5, 2]:
    pf_shove, pf_call = pf_table_combo_summary(depth)
    cfr_shove, cfr_call = solve_push_fold_cfr_demo(stack_bb=float(depth), iterations=500)
    print(f"  Depth {depth:>2d} bb | PF_TABLE Shove: {100*pf_shove:>5.1f}%, Call: {100*pf_call:>5.1f}% | CFR Demo Shove: {100*cfr_shove:>5.1f}%, Call: {100*cfr_call:>5.1f}%")

icm_demo = compute_bubble_metrics([2000.0, 5000.0, 150.0, 1850.0], hero_idx=0, opp_idx=1, to_call=1800.0, pot=2200.0, payouts=[0.50, 0.30, 0.20])
print(f"\nMalmuth-Harville Tournament ICM Demo: Hero Prize Equity = {100*icm_demo['icm_ev']:.1f}%, Bubble Factor = {icm_demo['bubble_factor']:.2f}x, cEV Need = {100*icm_demo['cev_need']:.1f}%, ICM Need = {100*icm_demo['icm_need']:.1f}%")

print("\nBayesian Opponent Profiling (Algorithm 19: Beta-Binomial 2-Sigma Gate on foldToBet):")
for k_folds, n_trials in [(2, 2), (5, 6), (12, 14), (20, 24)]:
    st = bayesian_stat("foldToBet", k_folds, n_trials)
    status = "GATED (z = 0, noise filtered)" if st["z"] == 0 else f"ACTIVE (z = {st['z']:+.3f})"
    print(f"  Observed {k_folds:>2d}/{n_trials:<2d} folds: Posterior Mean = {100*st['mean']:5.1f}% ± {100*st['sigma']:4.1f}% | {status}")

learn_buckets = {}
sample_trace = [{
    "hole": parse_cards("Qs Ts"),
    "board": parse_cards("Qh 8s 4c"),
    "opp_ids": [0, 3],
    "pot": 350,
    "to_call": 85,
    "action": "call",
    "key": "1:2:0:1",
}]
review_hand_trace(sample_trace, shown_hands={3: parse_cards("Ah Kd")}, learn_store=learn_buckets, bb=20)
print(f"\nHindsight-Free Self-Review (Algorithm 20): True Equity vs [Ah Kd + 1 unknown mucked] = {100*sample_trace[0]['true_eq']:.1f}%")

Exact Embedded PF_TABLE Nash Equilibrium Summary vs Simplified CFR Demo:
  Depth 20 bb | PF_TABLE Shove:  42.9%, Call:  22.0% | CFR Demo Shove:  37.1%, Call:  25.9%
  Depth 15 bb | PF_TABLE Shove:  51.2%, Call:  28.3% | CFR Demo Shove:  45.3%, Call:  39.3%
  Depth 10 bb | PF_TABLE Shove:  61.3%, Call:  40.4% | CFR Demo Shove:  49.5%, Call:  51.6%
  Depth  5 bb | PF_TABLE Shove:  75.1%, Call:  68.2% | CFR Demo Shove:  67.2%, Call:  88.3%
  Depth  2 bb | PF_TABLE Shove:  94.5%, Call: 100.0% | CFR Demo Shove:  88.3%, Call:  99.9%

Malmuth-Harville Tournament ICM Demo: Hero Prize Equity = 29.1%, Bubble Factor = 2.60x, cEV Need = 45.0%, ICM Need = 68.0%

Bayesian Opponent Profiling (Algorithm 19: Beta-Binomial 2-Sigma Gate on foldToBet):
  Observed  2/2  folds: Posterior Mean =  57.1% ± 12.8% | GATED (z = 0, noise filtered)
  Observed  5/6  folds: Posterior Mean =  61.1% ± 11.2% | GATED (z = 0, noise filtered)
  Observed 12/14 folds: Posterior Mean =  69.2% ±  8.9% | ACTIVE (z = +0.541)
  O

## 7. Verification of all seven invariants and a complete worked $4$-player hand

### 7.1 Core invariants verified on every transition

An invariant is a property that must remain true before and after every state transition. In `assert_engine_invariants` below, all seven formal invariants from our report are checked directly:

1. **Invariant $I_1$ (Chip conservation):** player stacks plus `state.pot` equal the initial table chip supply $M_0$, and during active play $\mathrm{state.pot} = \sum_p p.\mathrm{committed}$.
2. **Invariant $I_2$ (Stack and commitment bounds):** $p.\mathrm{stack} \ge 0$, $p.\mathrm{bet} \ge 0$, $p.\mathrm{committed} \ge p.\mathrm{bet}$, and `all_in` holds if and only if an un-folded player with positive commitment has $p.\mathrm{stack} = 0$.
3. **Invariant $I_3$ ($52$-card deck bijection):** the disjoint union of undealt stub cards, burn cards, community board cards, and dealt hole cards has cardinality $52$ and equals $\{0, \ldots, 51\}$.
4. **Invariant $I_4$ (Evaluator equivalence):** on every $5$, $6$, or $7$ card hand present on the table, `rank7fast` equals `best7`, and $\mathrm{Encode}(\mathrm{Decode}(v)) = v$.
5. **Invariant $I_5$ (Turn-order soundness):** whenever `acting_idx != -1`, the designated seat is dealt in, un-folded, not all-in, has positive stack, and owes an action.
6. **Invariant $I_6$ (Side-pot eligibility soundness):** horizontal pot slices sum to `state.pot`, no folded player is eligible, and every eligible player satisfies $p.\mathrm{committed} \ge L_j$.
7. **Invariant $I_7$ (Statistical gating and hindsight isolation):** small-sample observations inside two standard deviations ($|\mu - \mu_0| \le 2\sigma$) produce $z = 0$.

### 7.2 Walkthrough setup on a $4$-player table

We trace a complete hand on a $4$-player table (the minimum starting table size in our engine) with small blind $\mathrm{sb} = 10$ and big blind $\mathrm{bb} = 20$. Suppose Seat 0 posted the big blind on the prior hand (`bb_seat = 0`). Under TDA Rule 19, `start_hand` places the small blind at Seat 0 ($10$ chips), advances the big blind to Seat 1 ($20$ chips), places the button at Seat 3, and badges Seat 2 as UTG.

We arrange the tail of `custom_deck` so that normal dealing gives Seat 0 $\text{J}\spadesuit\,\text{J}\heartsuit$, Seat 1 $\text{Q}\spadesuit\,\text{T}\spadesuit$, Seat 2 $7\diamondsuit\,2\clubsuit$, and Seat 3 $\text{A}\heartsuit\,\text{K}\diamondsuit$, followed by burn cards and the board $\text{Q}\heartsuit\,8\spadesuit\,4\clubsuit\,2\diamondsuit\,7\clubsuit$. Seat 2 folds UTG, Seat 3 raises to $55$ on the button, and Seats 0 and 1 call. On the flop $\text{Q}\heartsuit\,8\spadesuit\,4\clubsuit$, Seat 0 checks, Seat 1 bets $100$, and Seat 3 shoves all-in for $185$ (a short raise of $+85$, which is less than the $100$ full raise increment). Seat 0 checked at $0$ and therefore has full raising rights ($185 - 0 \ge 100$), whereas Seat 1 already acted at $100$ and cannot reopen raising ($185 - 100 = 85 < 100$). Seat 0 folds, Seat 1 calls the $85$, and the $535$-chip pot is settled at showdown.

In [31]:
def assert_engine_invariants(state, initial_total_chips):
    "Verify all seven formal engine invariants (I1 through I7) on the current TableState."
    current_stacks = sum(p.stack for p in state.players)
    assert current_stacks + state.pot == initial_total_chips, (
        f"Invariant I1 violated: stacks ({current_stacks}) + pot ({state.pot}) != {initial_total_chips}"
    )
    if state.pot > 0:
        assert sum(p.committed for p in state.players) == state.pot, "Invariant I1b violated: sum(committed) != pot"

    for p in state.players:
        assert p.stack >= 0 and p.bet >= 0 and p.committed >= p.bet, "Invariant I2 violated: negative stack or bet"
        if p.dealt_in and not p.folded and p.committed > 0 and state.pot > 0:
            assert p.all_in == (p.stack == 0), f"Invariant I2b violated for {p.name}"

    all_cards = list(state.deck) + list(state.burned) + list(state.board)
    for p in state.players:
        if p.dealt_in:
            all_cards.extend(p.hole)
    assert len(all_cards) == 52 and set(all_cards) == set(FULL_DECK), "Invariant I3 violated: 52-card bijection broken"

    if len(state.board) >= 3:
        for p in state.players:
            if p.dealt_in and len(p.hole) == 2:
                cards_n = p.hole + state.board
                v_fast = rank7fast(cards_n)
                v_best, _ = best7(cards_n)
                assert v_fast == v_best, "Invariant I4 violated: rank7fast != best7"
                cat, ks = decode_score(v_fast)
                assert encode_score(cat, ks) == v_fast, "Invariant I4b violated: encode(decode(v)) != v"

    if state.acting_idx != -1:
        actor = state.players[state.acting_idx]
        assert actor.dealt_in and not actor.folded and not actor.all_in and actor.stack > 0, "Invariant I5 violated"
        assert (not actor.has_acted) or (actor.bet < state.current_bet), "Invariant I5b violated"

    if state.pot > 0:
        slices = compute_pots(state)
        assert sum(s.amount for s in slices) == state.pot, "Invariant I6 violated: slice sum != pot"
        for s in slices:
            for eid in s.eligible_ids:
                ep = state.players[eid]
                assert not ep.folded and ep.committed >= s.level, "Invariant I6b violated"

    assert bayesian_stat("foldToBet", 1, 2)["z"] == 0.0, "Invariant I7 violated"

print("Supported Table Sizes (4, 5, 6, 7, 8 Players Only) and Position Badges:")
for n_seats in VALID_PLAYER_COUNTS:
    t_players = [PlayerState(i, f"P{i}", stack=1500) for i in range(n_seats)]
    t_state = TableState(players=t_players, sb=10, bb=20, bb_seat=-1)
    start_hand(t_state)
    pos_map = position_labels(t_state)
    ordered_badges = [f"Seat {i}:{pos_map[i]}" for i in range(n_seats)]
    print(f"  {n_seats}-Player Table: {', '.join(ordered_badges)}")

walkthrough_players = [
    PlayerState(0, "Seat 0 (TAG)",   stack=1000, persona="TAG"),
    PlayerState(1, "Seat 1 (Human)", stack=1000, persona="SOLID", is_human=True),
    PlayerState(2, "Seat 2 (NIT)",   stack=1000, persona="NIT"),
    PlayerState(3, "Seat 3 (LAG)",   stack=240,  persona="LAG"),
]
w_state = TableState(players=walkthrough_players, sb=10, bb=20, bb_seat=0)
initial_M0 = sum(p.stack for p in w_state.players)

# Arrange the deck tail so pop() deals two rounds starting at sb_seat=0, then burns and deals the board.
deal_sequence = parse_cards("Js Qs 7d Ah Jh Ts 2c Kd 3s Qh 8s 4c 4s 2d 5s 7c")
custom_deck = [c for c in FULL_DECK if c not in deal_sequence] + list(reversed(deal_sequence))

start_hand(w_state, custom_deck=custom_deck)
assert [p.hole for p in w_state.players] == [
    parse_cards("Js Jh"),
    parse_cards("Qs Ts"),
    parse_cards("7d 2c"),
    parse_cards("Ah Kd"),
]
assert_engine_invariants(w_state, initial_M0)

print("\nSTEP 1: StartHand and TDA Rule 19 Dead-Button Blind Advancement (Algorithm 10)")
print(f"  Positions : {position_labels(w_state)}")
print(f"  Hole Cards: Seat 0 = {hand_str(w_state.players[0].hole)} (Chen={chen_score(*w_state.players[0].hole):.0f}), "
      f"Seat 1 = {hand_str(w_state.players[1].hole)} (Chen={chen_score(*w_state.players[1].hole):.0f}), "
      f"Seat 2 = {hand_str(w_state.players[2].hole)} (Chen={chen_score(*w_state.players[2].hole):.0f}), "
      f"Seat 3 = {hand_str(w_state.players[3].hole)} (Chen={chen_score(*w_state.players[3].hole):.0f})")
print(f"  First Actor Preflop: Seat {w_state.acting_idx} (UTG) | Pot = {w_state.pot}")

print("\nSTEP 2: Preflop Betting Round (Algorithms 11 and 12)")
apply_action(w_state, w_state.players[2], "fold")
apply_action(w_state, w_state.players[3], "raise", target_to=55)
apply_action(w_state, w_state.players[0], "call")
apply_action(w_state, w_state.players[1], "call")
assert w_state.acting_idx == -1
assert_engine_invariants(w_state, initial_M0)
print(f"  Preflop Complete: Pot = {w_state.pot}, Committed = {[p.committed for p in w_state.players]}")

print("\nSTEP 3: Flop Q♥ 8♠ 4♣ and Short All-In Reopening Rule (Algorithm 12)")
w_state.stage = "flop"
w_state.burned.append(w_state.deck.pop())
w_state.board = [w_state.deck.pop() for _ in range(3)]
assert w_state.board == parse_cards("Qh 8s 4c")
for p in w_state.players:
    p.bet = 0
    p.has_acted = False
    p.acted_at_bet = -1
    p.min_raise_seen = w_state.bb
w_state.current_bet = 0
w_state.last_raise = w_state.bb
w_state.raises = 0
w_state.acting_idx = find_next_actor(w_state, w_state.btn_seat)
assert_engine_invariants(w_state, initial_M0)

apply_action(w_state, w_state.players[0], "check")
apply_action(w_state, w_state.players[1], "raise", target_to=100)
apply_action(w_state, w_state.players[3], "raise", target_to=185)

print("  After Seat 3 short all-in to 185 (+85, less than last_raise=100):")
print(f"    Can Seat 0 (who checked at 0) reopen raising? {can_reopen(w_state, w_state.players[0])} (185 - 0 >= 100)")
print(f"    Can Seat 1 (who bet 100) reopen raising?      {can_reopen(w_state, w_state.players[1])} (185 - 100 = 85 < 100)")

apply_action(w_state, w_state.players[0], "fold")
assert not can_reopen(w_state, w_state.players[1])
apply_action(w_state, w_state.players[1], "call")
assert w_state.acting_idx == -1
assert_engine_invariants(w_state, initial_M0)
print(f"  Flop Complete: Pot = {w_state.pot} (Seat 0 folded 55, Seat 1 and Seat 3 committed 240 each)")

print("\nSTEP 4: Turn 2♦ and River 7♣ Runout plus Showdown Settlement (Algorithms 13, 14, 15)")
w_state.burned.append(w_state.deck.pop())
w_state.board.append(w_state.deck.pop())
w_state.burned.append(w_state.deck.pop())
w_state.board.append(w_state.deck.pop())
assert w_state.board == parse_cards("Qh 8s 4c 2d 7c")
assert_engine_invariants(w_state, initial_M0)

settlement = award_pots(w_state)
assert_engine_invariants(w_state, initial_M0)

for res in settlement:
    print(f"  Pot {res['pot_index']} ({res['amount']} chips) awarded to: {res['winners']}")
for p in w_state.players:
    hand_desc = format_hand_name(p.score) if p.score >= 0 else "Folded"
    print(f"  {p.name:<15}: Final Stack = {p.stack:>4d} | Showdown = {hand_desc} ({p.score})")
print(f"\nAll 7 Formal Engine Invariants (I1 to I7) Verified: Total Table Chips = {sum(p.stack for p in w_state.players)} == M_0 ({initial_M0})!")

Supported Table Sizes (4, 5, 6, 7, 8 Players Only) and Position Badges:
  4-Player Table: Seat 0:BTN, Seat 1:SB, Seat 2:BB, Seat 3:UTG
  5-Player Table: Seat 0:BTN, Seat 1:SB, Seat 2:BB, Seat 3:UTG, Seat 4:CO
  6-Player Table: Seat 0:BTN, Seat 1:SB, Seat 2:BB, Seat 3:UTG, Seat 4:HJ, Seat 5:CO
  7-Player Table: Seat 0:BTN, Seat 1:SB, Seat 2:BB, Seat 3:UTG, Seat 4:UTG+1, Seat 5:HJ, Seat 6:CO
  8-Player Table: Seat 0:BTN, Seat 1:SB, Seat 2:BB, Seat 3:UTG, Seat 4:UTG+1, Seat 5:MP, Seat 6:HJ, Seat 7:CO

STEP 1: StartHand and TDA Rule 19 Dead-Button Blind Advancement (Algorithm 10)
  Positions : {1: 'BB', 0: 'SB', 3: 'BTN', 2: 'UTG'}
  Hole Cards: Seat 0 = J♠ J♥ (Chen=12), Seat 1 = Q♠ T♠ (Chen=8), Seat 2 = 7♦ 2♣ (Chen=-1), Seat 3 = A♥ K♦ (Chen=10)
  First Actor Preflop: Seat 2 (UTG) | Pot = 30

STEP 2: Preflop Betting Round (Algorithms 11 and 12)
  Preflop Complete: Pot = 165, Committed = [55, 55, 0, 55]

STEP 3: Flop Q♥ 8♠ 4♣ and Short All-In Reopening Rule (Algorithm 12)
  After Seat 3 sho

## 8. Final interpretation and summary of guarantees

### What our implementation verifies end to end

* Compact $0, \ldots, 51$ card integers, unbiased rejection-sampled Fisher-Yates shuffling, and our closure-isolated `DealerVault` with a salted SHA-256 commitment seal.
* A transparent $21$-subset evaluator (`best7`) paired with a single-pass bitmask evaluator (`rank7fast`) and a $5{,}000$-hand randomized differential test.
* Monte Carlo pot-share equity with unbiased Bessel ($N - 1$) standard error, multi-way range conditioning with preflop card-removal bunching, river board-lock chop detection, and exact $45{,}540$-deal heads-up turn enumeration.
* $4$, $5$, $6$, $7$, and $8$ player table support, TDA Rule 19 dead-button blind progression, cumulative short all-in reopening, horizontal side-pot slicing with direct contributor refunds on empty bands, and clockwise odd-chip distribution.
* Embedded $169 \times 19$ heads-up Nash push/fold lookup (`PF_TABLE`), recursive Malmuth-Harville tournament ICM bubble factors, $2\sigma$-gated Beta-Binomial opponent reads, and hindsight-free post-showdown review conditioned on revealed showdown hands (`equity_vs_known`).

## Appendix A: optional export utility

The helper below converts a `# %%` percent-formatted `.py` script into a standard v4 `.ipynb` Jupyter notebook.

In [30]:
def export_this_file_to_ipynb(source_py_path, output_ipynb_path="no_limit_holdem_engine_algorithms.ipynb"):
    "Convert a # %% percent-formatted Python script into a standard v4 .ipynb notebook."
    with open(source_py_path, "r", encoding="utf-8") as f:
        raw_lines = f.read().splitlines()

    nb_cells = []
    cur_type = None
    cur_lines = []

    def flush_cell():
        nonlocal cur_type, cur_lines
        if cur_type is None:
            return
        while cur_lines and not cur_lines[-1].strip():
            cur_lines.pop()
        formatted = [l + "\n" for l in cur_lines]
        if formatted:
            formatted[-1] = formatted[-1].rstrip("\n")
        if cur_type == "markdown":
            nb_cells.append({"cell_type": "markdown", "metadata": {}, "source": formatted})
        else:
            nb_cells.append({"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": formatted})
        cur_type = None
        cur_lines = []

    for line in raw_lines:
        if line.startswith("# %% [markdown]"):
            flush_cell()
            cur_type = "markdown"
        elif line.startswith("# %%"):
            flush_cell()
            cur_type = "code"
        elif cur_type == "markdown":
            cur_lines.append(line[2:] if line.startswith("# ") else (line[1:] if line.startswith("#") else line))
        elif cur_type == "code":
            cur_lines.append(line)
    flush_cell()

    nb_doc = {
        "cells": nb_cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.10"}
        },
        "nbformat": 4,
        "nbformat_minor": 5
    }
    with open(output_ipynb_path, "w", encoding="utf-8") as out_f:
        _json.dump(nb_doc, out_f, indent=2, ensure_ascii=False)
    return output_ipynb_path